Jupyter Notebook to extract closure quantities from Basilisk C DNS. 

There are two types of files.

    1. snapshot_XXXXX.h5 files which contain u,p,f,d data at t and t-dt along with parameters. 
    2. interface and interface_minus files which contain PLIC reconstructed interface data and curvatures at t and t-dt respectively.
The objective is to extract relevant closure quantities and spatial derivatives of thin-film equation terms from the DNS data.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from __future__ import annotations

import argparse
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path

HDF5 dataset loading functions

     1. load_h5 - Accepts a complete hdf5 file path of a directory and filename. Recursively loads every dataset into a dictionary whose keys are complete HDF5 paths. 
     2. inspect_h5 - Takes file path and prints all groups, datsets, dataset shapes, data types and root attributes.
     3. 

In [ ]:
def load_h5(filepath: str | Path, filename: str) -> dict[str, np.ndarray]:
    """
    Load all datasets from an HDF5 file.

    Parameters
    ----------
    filepath
        Directory containing the HDF5 file.

    filename
        Name of the HDF5 file, including the .h5 extension.

    Returns
    -------
    dict[str, np.ndarray]
        Dictionary containing all HDF5 datasets.

        The dictionary keys are the full dataset paths inside the
        HDF5 file. The values are NumPy arrays.

    Raises
    ------
    FileNotFoundError
        If the requested HDF5 file does not exist.

    OSError
        If the file exists but cannot be opened as an HDF5 file.
    """
    full_path = Path(filepath) / filename

    if not full_path.is_file():
        raise FileNotFoundError(
            f"HDF5 file was not found: {full_path.resolve()}"
        )

    data: dict[str, np.ndarray] = {}

    with h5py.File(full_path, mode="r") as h5_file:

        def load_dataset(name: str, obj: h5py.Dataset | h5py.Group) -> None:
            if isinstance(obj, h5py.Dataset):
                data[name] = obj[...]

        h5_file.visititems(load_dataset)

    return data

In [ ]:
def compute_cell_centers(
    points: np.ndarray,
    topology: np.ndarray,
) -> np.ndarray:
    """
    Compute cell-center coordinates from mesh vertices and topology.

    Parameters
    ----------
    points
        Vertex coordinates with shape (n_points, n_dimensions).

    topology
        Cell connectivity with shape (n_cells, n_vertices_per_cell).
        Each row contains the vertex indices belonging to one cell.

    Returns
    -------
    np.ndarray
        Cell-center coordinates with shape (n_cells, n_dimensions).
    """
    points = np.asarray(points)
    topology = np.asarray(topology, dtype=np.int64)

    # Coordinates of every vertex belonging to every cell.
    # Shape:
    # (n_cells, n_vertices_per_cell, n_dimensions)
    cell_vertices = points[topology]

    # Average the vertex coordinates for each cell.
    cell_centers = cell_vertices.mean(axis=1)

    return cell_centers

In [ ]:
def build_cartesian_cell_grid(
    cell_centers: np.ndarray,
    decimals: int = 12,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build Cartesian cell-center coordinate matrices.

    Parameters
    ----------
    cell_centers
        Cell-center coordinates with shape (n_cells, 2) or
        (n_cells, 3).

    decimals
        Number of decimal places used when identifying unique
        coordinates. This prevents floating-point roundoff from
        creating duplicate coordinate values.

    Returns
    -------
    x_matrix
        Cell-center x coordinates with shape (ny, nx).

    y_matrix
        Cell-center y coordinates with shape (ny, nx).

    sorting_indices
        Indices that reorder cell-based fields into matrix order.
    """
    cell_centers = np.asarray(cell_centers)

    x = cell_centers[:, 0]
    y = cell_centers[:, 1]

    x_rounded = np.round(x, decimals=decimals)
    y_rounded = np.round(y, decimals=decimals)

    unique_x = np.unique(x_rounded)
    unique_y = np.unique(y_rounded)

    nx = len(unique_x)
    ny = len(unique_y)

    if nx * ny != len(cell_centers):
        raise ValueError(
            "The cell centers do not form a complete Cartesian grid. "
            f"Found nx={nx}, ny={ny}, but nx*ny={nx * ny} and "
            f"n_cells={len(cell_centers)}."
        )

    # Primary sorting key: y
    # Secondary sorting key: x
    sorting_indices = np.lexsort((x_rounded, y_rounded))

    x_sorted = x[sorting_indices]
    y_sorted = y[sorting_indices]

    x_matrix = x_sorted.reshape(ny, nx)
    y_matrix = y_sorted.reshape(ny, nx)

    return x_matrix, y_matrix, sorting_indices

In [ ]:
def vector_to_matrices(
    vector: np.ndarray,
    sorting_indices: np.ndarray,
    matrix_shape: tuple[int, int],
) -> tuple[np.ndarray, ...]:
    """
    Convert a cell-based vector field into component matrices.

    Parameters
    ----------
    vector
        Vector field with shape (n_cells, n_components).

    sorting_indices
        Sorting indices returned by build_cartesian_cell_grid().

    matrix_shape
        Desired matrix shape, normally x_matrix.shape.

    Returns
    -------
    tuple[np.ndarray, ...]
        One matrix for each vector component.
    """
    vector = np.asarray(vector)

    if vector.ndim != 2:
        raise ValueError(
            f"Expected vector shape (n_cells, n_components), "
            f"but received {vector.shape}."
        )

    if vector.shape[0] != len(sorting_indices):
        raise ValueError(
            f"Vector contains {vector.shape[0]} cells, but the mesh contains "
            f"{len(sorting_indices)} cells."
        )

    vector_sorted = vector[sorting_indices]

    return tuple(
        vector_sorted[:, component].reshape(matrix_shape)
        for component in range(vector.shape[1])
    )

In [ ]:
def scalar_to_matrix(
    scalar: np.ndarray,
    sorting_indices: np.ndarray,
    matrix_shape: tuple[int, int],
) -> np.ndarray:
    """
    Convert a cell-based scalar field into a 2D Cartesian matrix.

    Parameters
    ----------
    scalar
        Scalar field with one value per cell.

    sorting_indices
        Sorting indices returned by build_cartesian_cell_grid().

    matrix_shape
        Desired matrix shape, normally x_matrix.shape.

    Returns
    -------
    np.ndarray
        Scalar field with shape (ny, nx).
    """
    scalar = np.asarray(scalar).squeeze()

    if scalar.ndim != 1:
        raise ValueError(
            f"Expected a one-dimensional scalar field after squeezing, "
            f"but received shape {scalar.shape}."
        )

    if len(scalar) != len(sorting_indices):
        raise ValueError(
            f"Scalar contains {len(scalar)} values, but the mesh contains "
            f"{len(sorting_indices)} cells."
        )

    return scalar[sorting_indices].reshape(matrix_shape)

The following functions are closure utilities aimed at using the cell-centered data to derive the required quantities.

In [ ]:
def depth_integral(
    x: np.ndarray,
    y: np.ndarray,
    field: np.ndarray,
    *,
    lower_boundary: float = 0.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Depth-integrate a cell-centered field along y at every x-location.

    Computes

        I(x) = integral field(x, y) dy

    using a finite-volume sum:

        I(x_i) = sum_j field[j, i] * dy[j, i]

    Parameters
    ----------
    x
        Cell-center x-coordinate matrix with shape (ny, nx).

    y
        Cell-center y-coordinate matrix with shape (ny, nx).

    field
        Cell-centered field with shape (ny, nx).

    lower_boundary
        Physical location of the lower boundary. For a wall at y = 0,
        use the default value 0.0.

    Returns
    -------
    x_locations
        One-dimensional array of x locations with shape (nx,).

    integral
        Depth-integrated field at every x location with shape (nx,).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    field = np.asarray(field, dtype=float)

    if x.ndim != 2 or y.ndim != 2 or field.ndim != 2:
        raise ValueError("x, y, and field must be two-dimensional arrays.")

    if x.shape != y.shape or x.shape != field.shape:
        raise ValueError(
            "x, y, and field must have identical shapes. "
            f"Received x={x.shape}, y={y.shape}, "
            f"field={field.shape}."
        )

    ny, nx = field.shape

    if ny < 2:
        raise ValueError(
            "At least two cells in the y-direction are required "
            "to reconstruct cell boundaries."
        )

    x_locations = x[0, :].copy()
    integral = np.zeros(nx, dtype=float)

    for i in range(nx):
        y_column = y[:, i]
        field_column = field[:, i]

        # Ensure cells are ordered from bottom to top.
        order = np.argsort(y_column)
        y_sorted = y_column[order]
        field_sorted = field_column[order]

        if np.any(np.diff(y_sorted) <= 0.0):
            raise ValueError(
                f"Repeated or non-increasing y coordinates found at "
                f"x-index {i}."
            )

        # Reconstruct cell-face locations from cell centers.
        y_faces = np.empty(ny + 1, dtype=float)

        # Physical lower wall.
        y_faces[0] = lower_boundary

        # Interior faces lie halfway between neighboring cell centers.
        y_faces[1:ny] = 0.5 * (
            y_sorted[:-1] + y_sorted[1:]
        )

        # Extrapolate the upper face by half the final center spacing.
        y_faces[ny] = (
            y_sorted[-1]
            + 0.5 * (y_sorted[-1] - y_sorted[-2])
        )

        cell_heights = np.diff(y_faces)

        if np.any(cell_heights <= 0.0):
            raise ValueError(
                f"Non-positive cell height reconstructed at x-index {i}."
            )

        integral[i] = np.sum(
            field_sorted * cell_heights
        )

    return x_locations, integral

In [ ]:
def compute_wall_shear(
    x: np.ndarray,
    y: np.ndarray,
    u: np.ndarray,
    mu: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute lower-wall shear stress from a cell-centered velocity field.

    Assumptions
    -----------
    - Uniform Cartesian mesh.
    - Lower wall is located at y = 0.
    - No-slip condition: u_wall = 0.
    - Cell centers are located at:
          y_j = (j + 1/2) * dy

    Parameters
    ----------
    x, y : ndarray, shape (ny, nx)
        Cell-center coordinates.

    u : ndarray, shape (ny, nx)
        Streamwise velocity.

    mu : float
        Dynamic viscosity.

    order : 4
        Accuracy order of the wall-gradient stencil.

    Returns
    -------
    x_wall : ndarray, shape (nx,)
        Streamwise coordinates along the wall.

    tau_w : ndarray, shape (nx,)
        Wall shear stress:
            tau_w = mu * (du/dy)_wall
    """

    if x.shape != y.shape or x.shape != u.shape:
        raise ValueError("x, y, and u must have identical shapes.")

    if u.ndim != 2:
        raise ValueError("x, y, and u must be two-dimensional arrays.")

    required_cells = {2: 2, 3: 3, 4: 4}

    order = 4
    if u.shape[0] < required_cells[order]:
        raise ValueError(
            f"Order {order} requires at least "
            f"{required_cells[order]} wall-normal cells."
        )

    dy = y[1, 0] - y[0, 0]

    if dy <= 0.0:
        raise ValueError(
            "The y coordinate must increase away from the lower wall."
        )

    u1 = u[0, :]
    u2 = u[1, :]
    u3 = u[2, :]
    u4 = u[3, :]

    dudy_wall = (
        3675.0 * u1
        - 1225.0 * u2
        + 441.0 * u3
        - 75.0 * u4
    ) / (840.0 * dy)

    tau_w = mu * dudy_wall

    return x[0, :], tau_w

In [ ]:
def compute_wall_shear_opt(
    x: np.ndarray,
    y: np.ndarray,
    u: np.ndarray,
    mu: float = 1.0,
    order: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute lower-wall shear stress from a cell-centered velocity field.

    Assumptions
    -----------
    - Uniform Cartesian mesh.
    - Lower wall is located at y = 0.
    - No-slip condition: u_wall = 0.
    - Cell centers are located at:
          y_j = (j + 1/2) * dy

    Parameters
    ----------
    x, y : ndarray, shape (ny, nx)
        Cell-center coordinates.

    u : ndarray, shape (ny, nx)
        Streamwise velocity.

    mu : float
        Dynamic viscosity.

    order : {2, 3, 4}
        Accuracy order of the wall-gradient stencil.

    Returns
    -------
    x_wall : ndarray, shape (nx,)
        Streamwise coordinates along the wall.

    tau_w : ndarray, shape (nx,)
        Wall shear stress:
            tau_w = mu * (du/dy)_wall
    """

    if x.shape != y.shape or x.shape != u.shape:
        raise ValueError("x, y, and u must have identical shapes.")

    if u.ndim != 2:
        raise ValueError("x, y, and u must be two-dimensional arrays.")

    required_cells = {2: 2, 3: 3, 4: 4}

    if order not in required_cells:
        raise ValueError("order must be 2, 3, or 4.")

    if u.shape[0] < required_cells[order]:
        raise ValueError(
            f"Order {order} requires at least "
            f"{required_cells[order]} wall-normal cells."
        )

    dy = y[1, 0] - y[0, 0]

    if dy <= 0.0:
        raise ValueError(
            "The y coordinate must increase away from the lower wall."
        )

    u1 = u[0, :]
    u2 = u[1, :]

    if order == 2:
        dudy_wall = (
            9.0 * u1
            - u2
        ) / (3.0 * dy)

    elif order == 3:
        u3 = u[2, :]

        dudy_wall = (
            225.0 * u1
            - 50.0 * u2
            + 9.0 * u3
        ) / (60.0 * dy)

    else:
        u3 = u[2, :]
        u4 = u[3, :]

        dudy_wall = (
            3675.0 * u1
            - 1225.0 * u2
            + 441.0 * u3
            - 75.0 * u4
        ) / (840.0 * dy)

    tau_w = mu * dudy_wall

    return x[0, :], tau_w


In [ ]:
def first_derivative_x2(
    x: np.ndarray,
    field: np.ndarray,
) -> np.ndarray:
    """
    Compute the first derivative of a one-dimensional field with respect to x.

    A second-order central-difference scheme is used at interior points.
    Second-order one-sided differences are used at the boundaries.

    Parameters
    ----------
    x : ndarray, shape (nx,)
        One-dimensional x coordinates.

    field : ndarray, shape (nx,)
        Field values, such as h(x) or q(x).

    Returns
    -------
    dfield_dx : ndarray, shape (nx,)
        First derivative of the field with respect to x.

    Notes
    -----
    Interior stencil:

        df/dx |_i = (f[i+1] - f[i-1]) / (2*dx)

    Left boundary:

        df/dx |_0 = (-3*f[0] + 4*f[1] - f[2]) / (2*dx)

    Right boundary:

        df/dx |_{N-1}
            = (3*f[N-1] - 4*f[N-2] + f[N-3]) / (2*dx)

    The grid is assumed to be uniformly spaced.
    """

    x = np.asarray(x, dtype=float)
    field = np.asarray(field, dtype=float)

    if x.ndim != 1 or field.ndim != 1:
        raise ValueError("x and field must be one-dimensional arrays.")

    if x.shape != field.shape:
        raise ValueError("x and field must have the same shape.")

    if x.size < 3:
        raise ValueError(
            "At least three x points are required for the "
            "second-order boundary stencils."
        )

    dx_values = np.diff(x)
    dx = dx_values[0]

    if dx <= 0.0:
        raise ValueError("x coordinates must increase monotonically.")

    if not np.allclose(dx_values, dx):
        raise ValueError(
            "The x grid is not uniformly spaced. "
            "This function assumes constant dx."
        )

    dfield_dx = np.empty_like(field, dtype=float)

    # Interior points: second-order central difference
    dfield_dx[1:-1] = (
        field[2:] - field[:-2]
    ) / (2.0 * dx)

    # Left boundary: second-order forward difference
    dfield_dx[0] = (
        -3.0 * field[0]
        + 4.0 * field[1]
        - field[2]
    ) / (2.0 * dx)

    # Right boundary: second-order backward difference
    dfield_dx[-1] = (
        3.0 * field[-1]
        - 4.0 * field[-2]
        + field[-3]
    ) / (2.0 * dx)

    return dfield_dx

In [ ]:
def first_derivative_x(
    x: np.ndarray,
    field: np.ndarray,
) -> np.ndarray:
    """
    Compute the first derivative of a one-dimensional field with respect to x.

    Fourth-order central differences are used in the interior.
    Fourth-order one-sided differences are used near the boundaries.

    Parameters
    ----------
    x : ndarray, shape (nx,)
        One-dimensional uniformly spaced x coordinates.

    field : ndarray, shape (nx,)
        Field values, such as h(x) or q(x).

    Returns
    -------
    dfield_dx : ndarray, shape (nx,)
        Fourth-order approximation to df/dx.

    Notes
    -----
    Interior stencil:

        f'_i =
            (f[i-2] - 8*f[i-1] + 8*f[i+1] - f[i+2])
            / (12*dx)

    The grid must be uniformly spaced.

    At least 5 points are required.
    """

    x = np.asarray(x, dtype=float)
    field = np.asarray(field, dtype=float)

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if x.ndim != 1 or field.ndim != 1:
        raise ValueError(
            "x and field must be one-dimensional arrays."
        )

    if x.shape != field.shape:
        raise ValueError(
            "x and field must have the same shape."
        )

    if x.size < 5:
        raise ValueError(
            "At least five x points are required for "
            "fourth-order differentiation."
        )

    dx_values = np.diff(x)
    dx = dx_values[0]

    if dx <= 0.0:
        raise ValueError(
            "x coordinates must increase monotonically."
        )

    if not np.allclose(dx_values, dx):
        raise ValueError(
            "The x grid is not uniformly spaced. "
            "This function assumes constant dx."
        )

    dfield_dx = np.empty_like(field, dtype=float)

    # --------------------------------------------------------
    # Interior
    # Fourth-order central difference
    # --------------------------------------------------------

    dfield_dx[2:-2] = (
        field[:-4]
        - 8.0 * field[1:-3]
        + 8.0 * field[3:-1]
        - field[4:]
    ) / (12.0 * dx)

    # --------------------------------------------------------
    # Left boundary
    # Fourth-order forward differences
    # --------------------------------------------------------

    dfield_dx[0] = (
        -25.0 * field[0]
        + 48.0 * field[1]
        - 36.0 * field[2]
        + 16.0 * field[3]
        - 3.0 * field[4]
    ) / (12.0 * dx)

    dfield_dx[1] = (
        -3.0 * field[0]
        - 10.0 * field[1]
        + 18.0 * field[2]
        - 6.0 * field[3]
        + field[4]
    ) / (12.0 * dx)

    # --------------------------------------------------------
    # Right boundary
    # Fourth-order backward differences
    # --------------------------------------------------------

    dfield_dx[-2] = (
        -field[-5]
        + 6.0 * field[-4]
        - 18.0 * field[-3]
        + 10.0 * field[-2]
        + 3.0 * field[-1]
    ) / (12.0 * dx)

    dfield_dx[-1] = (
        3.0 * field[-5]
        - 16.0 * field[-4]
        + 36.0 * field[-3]
        - 48.0 * field[-2]
        + 25.0 * field[-1]
    ) / (12.0 * dx)

    return dfield_dx

In [ ]:
# def second_derivative_x(
#     x: np.ndarray,
#     field: np.ndarray,
# ) -> np.ndarray:
#     """
#     Compute the second derivative of a one-dimensional field with
#     respect to x.

#     Uses:
#         - second-order central differences in the interior
#         - second-order one-sided differences at the boundaries

#     Parameters
#     ----------
#     x : ndarray (nx,)
#         Uniformly-spaced x coordinates.

#     field : ndarray (nx,)
#         Field values.

#     Returns
#     -------
#     d2field_dx2 : ndarray (nx,)
#         Second derivative.
#     """

#     x = np.asarray(x, dtype=float)
#     field = np.asarray(field, dtype=float)

#     if x.ndim != 1 or field.ndim != 1:
#         raise ValueError("x and field must be one-dimensional.")

#     if x.shape != field.shape:
#         raise ValueError("x and field must have the same shape.")

#     if x.size < 4:
#         raise ValueError(
#             "Need at least four points for the boundary stencil."
#         )

#     dx_values = np.diff(x)
#     dx = dx_values[0]

#     if not np.allclose(dx_values, dx):
#         raise ValueError("Grid must be uniformly spaced.")

#     d2field_dx2 = np.empty_like(field)

#     #
#     # Interior (second-order central)
#     #
#     d2field_dx2[1:-1] = (
#         field[2:]
#         - 2.0 * field[1:-1]
#         + field[:-2]
#     ) / dx**2

#     #
#     # Left boundary (second-order forward)
#     #
#     d2field_dx2[0] = (
#         2.0 * field[0]
#         - 5.0 * field[1]
#         + 4.0 * field[2]
#         - field[3]
#     ) / dx**2

#     #
#     # Right boundary (second-order backward)
#     #
#     d2field_dx2[-1] = (
#         2.0 * field[-1]
#         - 5.0 * field[-2]
#         + 4.0 * field[-3]
#         - field[-4]
#     ) / dx**2

#     return d2field_dx2

In [ ]:
def second_derivative_x(
    x: np.ndarray,
    field: np.ndarray,
) -> np.ndarray:
    """
    Compute the second derivative of a one-dimensional field
    with respect to x.

    Uses:
        - fourth-order central differences in the interior
        - fourth-order one-sided differences near the boundaries

    Parameters
    ----------
    x : ndarray, shape (nx,)
        Uniformly spaced x coordinates.

    field : ndarray, shape (nx,)
        Field values.

    Returns
    -------
    d2field_dx2 : ndarray, shape (nx,)
        Fourth-order approximation to the second derivative.
    """

    x = np.asarray(x, dtype=float)
    field = np.asarray(field, dtype=float)

    if x.ndim != 1 or field.ndim != 1:
        raise ValueError(
            "x and field must be one-dimensional."
        )

    if x.shape != field.shape:
        raise ValueError(
            "x and field must have the same shape."
        )

    if x.size < 6:
        raise ValueError(
            "At least six points are required for "
            "fourth-order boundary stencils."
        )

    dx_values = np.diff(x)
    dx = dx_values[0]

    if dx <= 0.0:
        raise ValueError(
            "x coordinates must increase monotonically."
        )

    if not np.allclose(dx_values, dx):
        raise ValueError(
            "Grid must be uniformly spaced."
        )

    d2field_dx2 = np.empty_like(
        field,
        dtype=float,
    )

    # --------------------------------------------------------
    # Interior: fourth-order central difference
    # --------------------------------------------------------

    d2field_dx2[2:-2] = (
        -field[4:]
        + 16.0 * field[3:-1]
        - 30.0 * field[2:-2]
        + 16.0 * field[1:-3]
        - field[:-4]
    ) / (12.0 * dx**2)

    # --------------------------------------------------------
    # Left boundary
    # Fourth-order one-sided differences
    # --------------------------------------------------------

    d2field_dx2[0] = (
        45.0 * field[0]
        - 154.0 * field[1]
        + 214.0 * field[2]
        - 156.0 * field[3]
        + 61.0 * field[4]
        - 10.0 * field[5]
    ) / (12.0 * dx**2)

    d2field_dx2[1] = (
        10.0 * field[0]
        - 15.0 * field[1]
        - 4.0 * field[2]
        + 14.0 * field[3]
        - 6.0 * field[4]
        + field[5]
    ) / (12.0 * dx**2)

    # --------------------------------------------------------
    # Right boundary
    # Mirror of left-boundary stencils
    # --------------------------------------------------------

    d2field_dx2[-2] = (
        field[-6]
        - 6.0 * field[-5]
        + 14.0 * field[-4]
        - 4.0 * field[-3]
        - 15.0 * field[-2]
        + 10.0 * field[-1]
    ) / (12.0 * dx**2)

    d2field_dx2[-1] = (
        -10.0 * field[-6]
        + 61.0 * field[-5]
        - 156.0 * field[-4]
        + 214.0 * field[-3]
        - 154.0 * field[-2]
        + 45.0 * field[-1]
    ) / (12.0 * dx**2)

    return d2field_dx2

In [ ]:
def first_derivative_t(
    field: np.ndarray,
    field_minus: np.ndarray,
    time: float,
    time_minus: float,
) -> np.ndarray:
    """
    Compute the first-order time derivative using two snapshots.

    Parameters
    ----------
    field : ndarray
        Quantity at the current time.

    field_minus : ndarray
        Quantity at the previous time.

    time : float
        Current time.

    time_minus : float
        Previous time.

    Returns
    -------
    dfield_dt : ndarray
        Time derivative

            dfield_dt = (field - field_minus) / (time - time_minus)

        with the same shape as the input arrays.
    """

    field = np.asarray(field, dtype=float)
    field_minus = np.asarray(field_minus, dtype=float)

    if field.shape != field_minus.shape:
        raise ValueError(
            "field and field_minus must have identical shapes. "
            f"Received {field.shape} and {field_minus.shape}."
        )

    dt = float(time) - float(time_minus)

    if dt <= 0.0:
        raise ValueError(
            f"time must be greater than time_minus. "
            f"Received time={time}, time_minus={time_minus}."
        )

    return (field - field_minus) / dt

Plot utilities

In [ ]:
def plot_scalar_contour(
    x: np.ndarray,
    y: np.ndarray,
    scalar: np.ndarray,
    *,
    title: str = "Scalar field",
    scalar_label: str = "Value",
    xlabel: str = "x",
    ylabel: str = "y",
    levels: int = 50,
    filled: bool = True,
    equal_aspect: bool = False,
    show: bool = True,
) -> tuple[Figure, Axes]:
    """
    Plot a two-dimensional scalar field on a cell-centered grid.

    Parameters
    ----------
    x, y
        Cell-center coordinate matrices with shape (ny, nx).

    scalar
        Cell-centered scalar matrix with shape (ny, nx).

    title
        Plot title.

    scalar_label
        Label for the color bar.

    xlabel, ylabel
        Axis labels.

    levels
        Number of contour levels.

    filled
        If True, use filled contours. If False, use contour lines.

    equal_aspect
        If True, use the same physical scaling in x and y.

    show
        If True, call matplotlib.pyplot.show().

    Returns
    -------
    tuple[Figure, Axes]
        Matplotlib figure and axes objects.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    scalar = np.asarray(scalar)

    if x.ndim != 2 or y.ndim != 2 or scalar.ndim != 2:
        raise ValueError(
            "x, y, and scalar must all be two-dimensional matrices."
        )

    if x.shape != y.shape or x.shape != scalar.shape:
        raise ValueError(
            "x, y, and scalar must have identical shapes. "
            f"Received x={x.shape}, y={y.shape}, "
            f"scalar={scalar.shape}."
        )

    if levels < 2:
        raise ValueError("levels must be at least 2.")

    figure, axes = plt.subplots()

    if filled:
        contour = axes.contourf(
            x,
            y,
            scalar,
            levels=levels,
        )
    else:
        contour = axes.contour(
            x,
            y,
            scalar,
            levels=levels,
        )
        axes.clabel(contour, inline=True)

    colorbar = figure.colorbar(contour, ax=axes)
    colorbar.set_label(scalar_label)

    axes.set_xlabel(xlabel)
    axes.set_ylabel(ylabel)
    axes.set_title(title)

    if equal_aspect:
        axes.set_aspect("equal", adjustable="box")

    figure.tight_layout()

    if show:
        plt.show()

    return figure, axes


def plot_profile(
    x: np.ndarray,
    y: np.ndarray,
    field: np.ndarray,
    x_location: float,
    xlabel: str = "Field",
    ylabel: str = "y",
    title: str | None = None,
    ax=None,
):
    """
    Plot the wall-normal profile of a field at the x location
    closest to the requested value.

    Parameters
    ----------
    x : ndarray (ny, nx)
        Cell-center x coordinates.

    y : ndarray (ny, nx)
        Cell-center y coordinates.

    field : ndarray (ny, nx)
        Scalar field.

    x_location : float
        Desired x coordinate.

    xlabel : str
        Label for x-axis (field).

    ylabel : str
        Label for y-axis.

    title : str, optional
        Figure title.

    ax : matplotlib.axes.Axes, optional
        Existing axes.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure object.

    ax : matplotlib.axes.Axes
        Axes object.

    column : int
        Column index used.
    """

    if x.shape != y.shape or x.shape != field.shape:
        raise ValueError("x, y and field must have identical shapes.")

    # Find nearest x-column
    x_coords = x[0, :]
    column = np.argmin(np.abs(x_coords - x_location))

    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 6))
    else:
        fig = ax.figure

    ax.plot(field[:, column], y[:, column], linewidth=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if title is None:
        ax.set_title(
            f"{xlabel} profile at x = {x_coords[column]:.5f}"
        )
    else:
        ax.set_title(title)

    ax.grid(True)

    return fig, ax, column

Main Code

In [ ]:
# t = 6000
t = 5200
case_dir = Path("/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/thinfilm-closure-ml/datasets/BasC/param_space/Re120_Ca0_010_theta45")

# t = 1800
# case_dir = Path("/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/thinfilm-closure-ml/datasets/BasC/param_space/Re200_Ca0_010_theta45")


output_dir = case_dir / "output"
interface_dir = case_dir / "infc"

snapshot_filename = f"snapshot_{t:06.0f}.h5"
interface_filename = f"interface_{t:06.0f}.h5"
interface_minus_filename = f"interface_minus_{t:06.0f}.h5"

snapshot_path = output_dir / snapshot_filename
interface_path = interface_dir / interface_filename
interface_minus_path = interface_dir / interface_minus_filename

required_files = [
    snapshot_path,
    interface_path,
    interface_minus_path,
]

missing_files = [path for path in required_files if not path.is_file()]

if missing_files:
    missing_text = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(
        "The following required files were not found:\n"
        f"{missing_text}"
    )

In [ ]:
flowfield_data = load_h5(
    output_dir,
    snapshot_filename,
)

interface_data = load_h5(
    interface_dir,
    interface_filename,
)

interface_minus_data = load_h5(
    interface_dir,
    interface_minus_filename,
)

In [ ]:
# print(f"Case directory:\n  {case_dir}\n")

# print(
#     f"Snapshot loaded:\n"
#     f"  {snapshot_path}\n"
#     f"  Number of datasets: {len(flowfield_data)}\n"
# )

# print(
#     f"Current interface loaded:\n"
#     f"  {interface_path}\n"
#     f"  Number of datasets: {len(interface_data)}\n"
# )

# print(
#     f"Previous interface loaded:\n"
#     f"  {interface_minus_path}\n"
#     f"  Number of datasets: {len(interface_minus_data)}"
# )

In [ ]:
print("Snapshot datasets:")
for key, value in flowfield_data.items():
    print(f"  {key:<35} shape={value.shape}, dtype={value.dtype}")

In [ ]:
print("\nCurrent interface datasets:")
for key, value in interface_data.items():
    print(f"  {key:<35} shape={value.shape}, dtype={value.dtype}")

In [ ]:
print("\nPrevious interface datasets:")
for key, value in interface_minus_data.items():
    print(f"  {key:<35} shape={value.shape}, dtype={value.dtype}")

Calculate cell centers first from points and topology

In [ ]:
points = flowfield_data["Geometry/Points"]
topology = flowfield_data["Topology"]
h0 = flowfield_data["Parameters/h0"]
Re = flowfield_data["Parameters/Re"]
Ca = flowfield_data["Parameters/Ca"]
beta = flowfield_data["Parameters/angle"]
# Compute one coordinate for every cell.
cell_centers = compute_cell_centers(points, topology)

# Construct Cartesian coordinate matrices.
x, y, sorting_indices = build_cartesian_cell_grid(cell_centers)

x = x
y = y

# Cell-centered scalar fields.
f = scalar_to_matrix(flowfield_data["Cells/f"], sorting_indices, x.shape)
d = scalar_to_matrix(flowfield_data["Cells/d"], sorting_indices, x.shape)
p = scalar_to_matrix(flowfield_data["Cells/p"], sorting_indices, x.shape)

fm = scalar_to_matrix(flowfield_data["Cells/f_minus"], sorting_indices, x.shape)
pm = scalar_to_matrix(flowfield_data["Cells/p_minus"], sorting_indices, x.shape)
## Non-dimensionalize pressure


t = flowfield_data["Parameters/time"]
tm = flowfield_data["Parameters/time_minus"]

# Cell-centered vector fields.
velocity_components = vector_to_matrices(
    flowfield_data["Cells/u.x"], 
    sorting_indices, x.shape)

u = velocity_components[0]
v = velocity_components[1]

velocity_components = vector_to_matrices(
        flowfield_data["Cells/u_minus.x"], 
        sorting_indices, x.shape)

um = velocity_components[0]
vm = velocity_components[1]

In [ ]:
xc, h = depth_integral(x,y,f)
_, q = depth_integral(x,y,u*f)
xc, hm = depth_integral(x,y,fm)
_, qm = depth_integral(x,y,um*fm)
_, shapefactor = depth_integral(x,y,u**2*f)
_, tau_w = compute_wall_shear(x,y,u, mu=1.0)

In [ ]:
# Second-order central-difference
h_x = first_derivative_x(xc, h)
h_x2 = first_derivative_x2(xc, h)
h_xx = second_derivative_x(xc, h)

q_x = first_derivative_x(xc, q)
q_xx = second_derivative_x(xc, q)

U_x = first_derivative_x(xc, q/h)
U_xx = second_derivative_x(xc, q/h)

In [ ]:
xlim1 = 600
xlim2 = 750
fig, ax = plt.subplots(
    2, 1,
    figsize=(12, 4),
    sharex=True,
    constrained_layout=True,
)
fig.suptitle(
    r"First derivatives of film thickness and flow rate",
    fontsize=14,
)
ax[0].plot(xc, h)
ax[0].set_title(r"$h$")
ax[0].set_xlabel(r"$x/h_0$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].grid(True)

ax[1].plot(xc, h_x)
ax[1].set_title(r"$h_x$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$h_x$")
ax[1].set_xlim(xlim1, xlim2)
ax[1].grid(True)


plt.show()

In [ ]:
xlim1 = 250
xlim2 = 400
fig, ax = plt.subplots(
    2, 1,
    figsize=(12, 4),
    sharex=True,
    constrained_layout=True,
)
fig.suptitle(
    r"First derivatives of film thickness and flow rate",
    fontsize=14,
)
ax[0].plot(xc, h)
ax[0].set_title(r"$h$")
ax[0].set_xlabel(r"$x/h_0$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].grid(True)

ax[1].plot(xc, h_xx)
ax[1].set_title(r"$h_{xx}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$h_{xx}$")
ax[1].set_xlim(xlim1, xlim2)
ax[1].grid(True)



plt.show()

In [ ]:
#Savitzky-Golay filter
from scipy.signal import savgol_filter

dx = xc[1] - xc[0]

h_x_savgol = savgol_filter(h, window_length=11, polyorder=3, deriv=1, delta=dx)
h_xx_savgol = savgol_filter(h, window_length=11, polyorder=3, deriv=2, delta=dx)


In [ ]:
poly_low, poly_high = 2, 4
window_low, window_high = 11, 21

settings = [
    (poly_low,  window_low),
    (poly_low,  window_high),
    (poly_high, window_low),
    (poly_high, window_high),
]

dx = xc[1] - xc[0]
xlim1, xlim2 = 376, 378

fig, ax = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True, constrained_layout=True
)

ax[0].plot(xc, h, color="black", linewidth=1.5, label="Original DNS")

for polyorder, window_length in settings:
    h_savgol = savgol_filter(
        h,
        window_length=window_length,
        polyorder=polyorder,
    )
    h_xx_savgol = savgol_filter(
        h,
        window_length=window_length,
        polyorder=polyorder,
        deriv=2,
        delta=dx,
    )

    label = f"p = {polyorder}, window = {window_length}"

    line, = ax[0].plot(xc, h_savgol, label=label)
    ax[1].plot(
        xc, h_xx_savgol,
        color=line.get_color(),
        label=label,
    )

ax[0].set_ylabel(r"$h$")
ax[0].set_title("Original and Savitzky–Golay filtered film thickness")
ax[0].legend()
ax[0].grid(True)

ax[1].set_ylabel(r"$h_{xx}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_title("Savitzky–Golay second derivative")
ax[1].legend()
ax[1].grid(True)
ax[1].set_xlim(xlim1, xlim2)

plt.show()

In [ ]:
from pybaselines.utils import whittaker_smooth
import numpy as np

h_wh = whittaker_smooth(h, lam=100, diff_order=2)
h_x_wh = np.gradient(h_wh, dx, edge_order=2)
h_xx_wh = np.gradient(h_x_wh, dx, edge_order=2)

In [ ]:
lam_low, lam_high = 10, 1000
diff_order_low, diff_order_high = 2, 3

settings = [
    (lam_low,  diff_order_low),
    (lam_low,  diff_order_high),
    (lam_high, diff_order_low),
    (lam_high, diff_order_high),
]

dx = xc[1] - xc[0]
xlim1, xlim2 = 375, 380

fig, ax = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True, constrained_layout=True
)

ax[0].plot(xc, h, color="black", linewidth=4.5, label="Original DNS")

for lam, diff_order in settings:
    h_wh = whittaker_smooth(h, lam=lam, diff_order=diff_order)
    h_x_wh = np.gradient(h_wh, dx, edge_order=2)
    h_xx_wh = np.gradient(h_x_wh, dx, edge_order=2)

    label = fr"$\lambda={lam}$, difference order $={diff_order}$"

    line, = ax[0].plot(xc, h_wh, label=label)
    ax[1].plot(xc, h_xx_wh, color=line.get_color(), label=label)

ax[0].set_ylabel(r"$h$")
ax[0].set_title("Original and Whittaker-smoothed film thickness")
ax[0].legend()
ax[0].grid(True)

ax[1].set_ylabel(r"$h_{xx}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_title("Second derivative after Whittaker smoothing")
ax[1].legend()
ax[1].grid(True)
ax[1].set_xlim(xlim1, xlim2)

plt.show()

In [ ]:
lam = 1000
diff_order = 3
m = 4
w = 31

dx = xc[1] - xc[0]
xlim1, xlim2 = 375, 380

if w % 2 != 1 or w <= m:
    raise ValueError("w must be odd and greater than m.")

# Filter h
h_wh = whittaker_smooth(h, lam=lam, diff_order=diff_order)
h_savgol = savgol_filter(h, window_length=w, polyorder=m)

# Calculate second derivatives
h_x_wh = np.gradient(h_wh, dx, edge_order=2)
h_xx_wh = np.gradient(h_x_wh, dx, edge_order=2)

h_xx_savgol = savgol_filter(
    h, window_length=w, polyorder=m, deriv=2, delta=dx
)

fig, ax = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True, constrained_layout=True
)

ax[0].plot(xc, h, color="black", linewidth=1.5, label="Original DNS")
ax[0].plot(
    xc, h_wh, "--", linewidth=2,
    label=fr"Whittaker: $\lambda={lam}$, difference order $={diff_order}$"
)
ax[0].plot(
    xc, h_savgol, ":", linewidth=2.5,
    label=fr"Savitzky–Golay: $m={m}$, $w={w}$"
)
ax[0].set_ylabel(r"$h$")
ax[0].set_title("Original and filtered film thickness")
ax[0].legend()
ax[0].grid(True)

ax[1].plot(
    xc, h_xx_wh,
    label=fr"Whittaker: $\lambda={lam}$, difference order $={diff_order}$"
)
ax[1].plot(
    xc, h_xx_savgol, "--",
    label=fr"Savitzky–Golay: $m={m}$, $w={w}$"
)
ax[1].set_ylabel(r"$h_{xx}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_title("Second derivative comparison")
ax[1].legend()
ax[1].grid(True)
ax[1].set_xlim(xlim1, xlim2)

plt.show()

In [ ]:
xlim1, xlim2 = 0, 800

savgol_settings = [(4, 11), (4, 21), (8, 11), (8, 21)]
whittaker_settings = [(10, 2), (10, 4), (1000, 6), (1000, 10)]

mask = (xc >= xlim1) & (xc <= xlim2)
if not np.any(mask):
    raise ValueError("No xc points fall within the selected x range.")

fig, ax = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True, constrained_layout=True
)

results = []

for polyorder, window_length in savgol_settings:
    h_filtered = savgol_filter(
        h, window_length=window_length, polyorder=polyorder
    )
    difference = h_filtered - h
    label = f"SavGol: p={polyorder}, window={window_length}"

    ax[0].plot(xc[mask], difference[mask], label=label)
    results.append((
        label,
        np.sqrt(np.mean(difference[mask] ** 2)),
        np.max(np.abs(difference[mask])),
    ))

for lam, diff_order in whittaker_settings:
    h_filtered = whittaker_smooth(
        h, lam=lam, diff_order=diff_order
    )
    difference = h_filtered - h
    label = f"Whittaker: λ={lam}, order={diff_order}"

    ax[1].plot(xc[mask], difference[mask], label=label)
    results.append((
        label,
        np.sqrt(np.mean(difference[mask] ** 2)),
        np.max(np.abs(difference[mask])),
    ))

for axis, title in zip(ax, ["Savitzky–Golay", "Whittaker"]):
    axis.axhline(0, color="black", linewidth=0.8)
    axis.set_title(title)
    axis.set_ylabel(r"$h_{\mathrm{filtered}}-h_{\mathrm{DNS}}$")
    axis.legend()
    axis.grid(True)

ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_xlim(xlim1, xlim2)
plt.show()

print(f"Height differences for {xlim1} ≤ x ≤ {xlim2}")
print(f"{'Filter':<38} {'RMSE':>14} {'Max |difference|':>18}")
for label, rmse, max_abs in results:
    print(f"{label:<38} {rmse:>14.6e} {max_abs:>18.6e}")

Now, we write a function to choose the best SavGol filter configuration

In [ ]:
def best_savgol_filter(
    x,
    values,
    m_values,
    p_values,
    metric="rmse",
    power=4,
):
    """
    Search Savitzky–Golay settings by the difference between filtered
    and input values across the full array.

    m: odd window length
    p: polynomial order
    metric: "rmse", "mae", "maxae", or "lp"
    power: exponent used when metric="lp"
    """
    x = np.asarray(x)
    values = np.asarray(values)

    if x.ndim != 1 or values.ndim != 1 or x.shape != values.shape:
        raise ValueError("x and values must be 1D arrays of the same length.")
    if not np.all(np.isfinite(values)):
        raise ValueError("values must contain only finite numbers.")
    if metric not in {"rmse", "mae", "maxae", "lp"}:
        raise ValueError("metric must be 'rmse', 'mae', 'maxae', or 'lp'.")
    if metric == "lp" and power < 1:
        raise ValueError("power must be at least 1.")

    results = []

    for m in m_values:
        for p in p_values:
            # p >= 2 permits using the selected settings for a second derivative.
            if m < 3 or m % 2 == 0 or m > len(values) or p < 2 or p >= m:
                continue

            filtered_values = savgol_filter(
                values, window_length=m, polyorder=p
            )
            difference = np.abs(filtered_values - values)

            if metric == "rmse":
                error = np.sqrt(np.mean(difference**2))
            elif metric == "mae":
                error = np.mean(difference)
            elif metric == "maxae":
                error = np.max(difference)
            else:  # metric == "lp"
                error = np.mean(difference**power)**(1 / power)

            results.append((error, m, p))

    if not results:
        raise ValueError("No valid (m, p) combinations were supplied.")

    results.sort()  # Lowest error first
    best_error, best_m, best_p = results[0]
    return best_m, best_p, best_error, results

In [ ]:
m_values = range(11, 42, 2)
p_values = range(2, 4)

best_m_h, best_p_h, best_error_h, results_h = best_savgol_filter(xc, h, m_values, p_values, metric="rmse", power=4)
best_m_q, best_p_q, best_error_q, results_q = best_savgol_filter(xc, q, m_values, p_values, metric="rmse", power=4)


print(f"h: Best m={best_m_h}, p={best_p_h}, L4 error={best_error_h:.6e}")
print(f"q: Best m={best_m_q}, p={best_p_q}, L4 error={best_error_q:.6e}")

In [ ]:
#Savitzky-Golay filter
from scipy.signal import savgol_filter

dx = xc[1] - xc[0]

h_filtered = savgol_filter(h, window_length=best_m_h, polyorder=best_p_h, deriv=0, delta=dx)
h_x = savgol_filter(h, window_length=best_m_h, polyorder=best_p_h, deriv=1, delta=dx)
h_xx = savgol_filter(h, window_length=best_m_h, polyorder=best_p_h, deriv=2, delta=dx)

q_filtered = savgol_filter(q, window_length=best_m_q, polyorder=best_p_q, deriv=0, delta=dx)
q_x = savgol_filter(q, window_length=best_m_q, polyorder=best_p_q, deriv=1, delta=dx)
q_xx = savgol_filter(q, window_length=best_m_q, polyorder=best_p_q, deriv=2, delta=dx)

In [ ]:
xlim1 = 375
xlim2 = 400
fig, ax = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True,
)
fig.suptitle(
    r"Derivatives of height",
    fontsize=14,
)
ax[0].plot(xc, h, linewidth = 4.5, label="Unfiltered")
ax[0].plot(xc, h_filtered, label="Filtered")
ax[0].set_title(r"$h$")
ax[0].set_xlabel(r"$x/h_0$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].legend()
ax[0].grid(True)

ax[1].plot(xc, h_x)
ax[1].set_title(r"$h_{x}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$h_{x}$")
ax[1].set_xlim(xlim1, xlim2)

ax[1].grid(True)


ax[2].plot(xc, h_xx)
ax[2].set_title(r"$h_{xx}$")
ax[2].set_xlabel(r"$x/h_0$")
ax[2].set_ylabel(r"$h_{xx}$")
ax[2].set_xlim(xlim1, xlim2)
ax[2].grid(True)


plt.show()

In [ ]:
xlim1 = 375
xlim2 = 400
fig, ax = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True,
)
fig.suptitle(
    r"Derivatives of flow rate",
    fontsize=14,
)
ax[0].plot(xc, q, linewidth = 4.5, label="Unfiltered")
ax[0].plot(xc, q_filtered, label="Filtered")
ax[0].set_title(r"$q$")
ax[0].set_xlabel(r"$x/h_0$")
ax[0].set_ylabel(r"$q$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].legend()
ax[0].grid(True)

ax[1].plot(xc, q_x)
ax[1].set_title(r"$q_{x}$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$q_{x}$")
ax[1].set_xlim(xlim1, xlim2)

ax[1].grid(True)


ax[2].plot(xc, q_xx)
ax[2].set_title(r"$q_{xx}$")
ax[2].set_xlabel(r"$x/h_0$")
ax[2].set_ylabel(r"$q_{xx}$")
ax[2].set_xlim(xlim1, xlim2)
ax[2].grid(True)


plt.show()

In [ ]:
t = flowfield_data["Parameters/time"]
tm = flowfield_data["Parameters/time_minus"]

h_t = first_derivative_t(h,hm,t,tm)
q_t = first_derivative_t(q,qm,t,tm)

Calculate curvature from Filtered quantities

In [ ]:
kappa_wh = (h_xx_wh) / (1 + h_x_wh**2) ** (3/2) 
kappa_savgol = (h_xx_savgol) / (1 + h_x_savgol**2) ** (3/2)
xlim1 = 600
xlim2 = 650
fig, ax = plt.subplots(
    3, 1,
    figsize=(12, 8),
    sharex=True,
    constrained_layout=True,
)
fig.suptitle(
    r"Film profile and curvature",
    fontsize=14,
)
ax[0].plot(xc, h)
ax[0].set_title(r"$h$")
ax[0].set_xlabel(r"$x/h_0$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].grid(True)

# ax[1].plot(xc, kappa, label="Unfiltered")
# ax[1].plot(xc, kappa_savgol, linestyle=":", label="SavGol")
ax[1].plot(xc, kappa_wh, linestyle="--",label="Curvature")
ax[1].plot(xc, h_xx_wh, linestyle="--",label="h_xx")
ax[1].set_title(r"$\kappa$")
ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$\kappa$")
ax[1].set_xlim(xlim1, xlim2)
# ax[1].set_ylim(-1, 1)
ax[1].legend()
ax[1].grid(True)

ax[2].plot(xc, h_x_wh, linestyle="--",label="h_xx")
ax[2].set_title(r"$h_x$")
ax[2].set_xlabel(r"$x/h_0$")
ax[2].set_ylabel(r"$h_x$")
ax[2].set_xlim(xlim1, xlim2)
ax[2].legend()
ax[2].grid(True)


plt.show()
# print(f"Minimum: {np.nanmin(kappa):.6e}")
# print(f"Maximum: {np.nanmax(kappa):.6e}")
# print(f"Mean:    {np.nanmean(np.abs(kappa)):.6e}")
# print(f"Median:  {np.nanmedian(np.abs(kappa)):.6e}")

In [ ]:
error_threshold = 0.05
absolute_error = np.abs((kappa_wh - h_xx_wh)/kappa_wh)

xlim1, xlim2 = 0, 800

# Find contiguous regions where the absolute error exceeds 0.1
above = absolute_error > error_threshold
edges = np.diff(np.r_[False, above, False].astype(int))
starts = np.flatnonzero(edges == 1)
stops = np.flatnonzero(edges == -1)  # exclusive

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(xc, h, color="steelblue", lw=1.5, label=r"$h$")

for j, (start, stop) in enumerate(zip(starts, stops)):
    ax.plot(
        xc[start:stop], h[start:stop],
        color="red", lw=3,
        label=r"$|\kappa-h_{xx}|>0.05$" if j == 0 else None,
    )

ax.set(xlabel=r"$x/h_0$", ylabel=r"$h$")
ax.set_xlim(xlim1, xlim2)
ax.grid(True)
ax.legend()
plt.show()

i_max = np.argmax(absolute_error)
print(
    f"Relative error: {absolute_error[i_max]:.6g} "
    f"at x = {xc[i_max]:.3f}"
)

for start, stop in zip(starts, stops):
    print(
        f"x = {xc[start]:.3f} to {xc[stop - 1]:.3f}: "
        f"maximum relative error = {absolute_error[start:stop].max():.6g}"
    )

In [ ]:
h_mask = (h > 1.6) | (h < 0.5)
counted = above & h_mask

n_above = np.count_nonzero(counted)
n_eligible = np.count_nonzero(h_mask)
percentage_above = 100 * n_above / n_eligible if n_eligible else 0.0

print(
    f"{n_above} of {n_eligible} xc values with h < 0.9 or h > 1.1 "
    f"({percentage_above:.2f}%) have absolute error > {error_threshold}"
)

The derivatives are a little noisy. We need to work on a filter or any alternate options to make it smooth and continuous.

The next section contains comparisons of the closure quantities and Lavalle's predicted values

In [ ]:
## Lavalle's closure terms
Re = flowfield_data["Parameters/Re"]
beta = flowfield_data["Parameters/angle"]
Fr2 = Re*np.sin(beta)/3
Lambda = 3.0



shapefactor_lavalle = 1.2* q**2 / h
tau_w_first_order = - (Re / 15.0) * Lambda**2 * h**4 * h_x
tau_w_zero_order = 3.0 * q / h**2

tau_w_lavalle = tau_w_zero_order + tau_w_first_order

R_tau = tau_w_first_order / tau_w_zero_order

In [ ]:
xlim1 = 0
xlim2 = 800
fig, ax = plt.subplots(
    3,
    1,
    figsize=(12, 9),
    sharex=True,
    constrained_layout=True,
)

fig.suptitle(
    "Closure quantities: DNS vs Lavalle",
    fontsize=14,
)

# Film thickness
ax[0].plot(
    xc,
    h,
    label="DNS",
)

ax[0].set_title(r"Film thickness, $h$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].grid(True)
ax[0].legend()

# Wall shear
ax[2].plot(
    xc,
    tau_w,
    label="DNS",
)

ax[2].plot(
    xc,
    tau_w_zero_order,
    "--",
    label=r"Lavalle - 0th order",
)

ax[2].plot(xc, tau_w_lavalle, ":", label=r"Lavalle")


# ax[1].set_title(rf"Wall shear, $\tau_w$ — MSE = {tau_w_mse:.3e}")
ax[2].set_ylabel(r"$\tau_w$")
ax[2].set_ylim(-1, 6)
ax[2].grid(True)
ax[2].set_xlim(xlim1, xlim2)
ax[2].legend()

# Shape factor
ax[1].plot(
    xc,
    shapefactor,
    label="DNS",
)

ax[1].plot(xc, shapefactor_lavalle, "--", label=r"Lavalle")

ax[1].set_xlabel(r"$x/h_0$")
ax[1].set_ylabel(r"$\Phi_{sf}$")
# ax[2].set_ylim(0, 4)
ax[1].grid(True)
ax[1].set_xlim(xlim1, xlim2)
ax[1].legend()

plt.show()

Other closure variables from the no-assumption thin film formulation

In [ ]:
def compute_interface_velocity(u, y, h):
    """
    Compute streamwise velocity u(x, h(x)) at the interface
    by linear interpolation in the y-direction.

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Cell-centered streamwise velocity.

    y : ndarray, shape (ny,nx)
        Cell-center y coordinates.

    h : ndarray, shape (nx,)
        Interface height at each x location.

    Returns
    -------
    u_interface : ndarray, shape (nx,)
        Streamwise velocity evaluated at y = h(x).
    """

    u = np.asarray(u)
    y = np.asarray(y)
    h = np.asarray(h)
    
    y = y[:,0]
    ny, nx = u.shape

    if y.size != ny:
        raise ValueError(
            f"y has {y.size} points but u has {ny} y-columns."
        )

    if h.size != nx:
        raise ValueError(
            f"h has {h.size} points but u has {nx} x-rows."
        )

    u_interface = np.empty(nx)

    for i in range(nx):
        u_interface[i] = np.interp(h[i], y, u[:, i])

    return u_interface

In [ ]:
Phi_i = compute_interface_velocity(u, y, h)

Phi_i_lavalle_zero = Lambda * h**2 / 2
Phi_i_lavalle_one = Re/24 * h * (-h**2 + 2* h**2 + 4* h**2) * h * ((Lambda * h) * Lambda * h_x)
Phi_i_lavalle = Phi_i_lavalle_zero + Phi_i_lavalle_one

In [ ]:
xlim1 = 0
xlim2 = 800
fig, ax = plt.subplots(
    3,
    1,
    figsize=(12, 6),
    sharex=True,
    constrained_layout=True,
)

fig.suptitle(
    "Higher order closure quantities: DNS vs Lavalle",
    fontsize=14,
)

# Film thickness
ax[0].plot(
    xc,
    h,
    label="DNS",
)

ax[0].set_title(r"Film thickness, $h$")
ax[0].set_ylabel(r"$h$")
ax[0].set_xlim(xlim1, xlim2)
ax[0].grid(True)
ax[0].legend()

# Velocity at interface
ax[1].plot(
    xc,
    Phi_i,
    label="DNS",
)

ax[1].plot(
    xc,
    Phi_i_lavalle_zero,
    "--",
    label=r"Lavalle - 0th order",
)

# ax[1].plot(
#     xc,
#     Phi_i_lavalle,
#     ":",
#     label=r"Lavalle",
# )


# ax[1].set_title(rf"Wall shear, $\tau_w$ — MSE = {tau_w_mse:.3e}")
ax[1].set_ylabel(r"$u_i$")
ax[1].axhline(1.5, linestyle="--",color="black")
ax[1].set_ylim(-1, 6)
ax[1].grid(True)
ax[1].set_xlim(xlim1, xlim2)
ax[1].legend()


ax[2].plot(
    xc,
    h * Phi_i / q,
    label="DNS",
)
# ax[1].plot(
#     xc,
#     Phi_i_lavalle,
#     ":",
#     label=r"Lavalle",
# )


# ax[1].set_title(rf"Wall shear, $\tau_w$ — MSE = {tau_w_mse:.3e}")
ax[2].set_ylabel(r"$\Phi_i$")
ax[2].set_ylim(-1, 3)
ax[2].grid(True)
ax[2].set_xlim(xlim1, xlim2)
ax[2].legend()


plt.show()

In [ ]:
xlim1 = 250
xlim2 = 400

fig, ax = plt.subplots(
    figsize=(12, 4),
    constrained_layout=True,
)

ax.plot(xc, h, color="tab:blue", label=r"Film thickness, $h$")
ax.plot(xc, q, color="tab:orange", label=r"Flow rate, $q$")
ax.plot(xc, Phi_i, color="tab:green", label=r"Interfacial velocity, $u_i$")

ax.axhline(
    1.5,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=r"$u_i = 1.5$",
)

ax.set_title(r"Film thickness, flow rate, and interfacial velocity")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$h,\ q,\ u_i$")
ax.set_xlim(xlim1, xlim2)
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

Now, we want to reconstruct Lavalle's methods.

Lavalle's momentum equation:

$\partial_t (hU) + \partial_x (hU^2 + P(h)) = \dfrac{1}{Re} \left(\Lambda h - \dfrac{3U}{h} - T \right) +$ Capillary term + Higher order terms


Generalized momentum equation:

$\partial_t (hU) + \partial_x (hU^2 \Phi_{sf} + \dfrac{h^2 cos(\beta)}{2 Fr^2}) = \dfrac{1}{Re} \left(\Lambda h - \tau_w \right) +$ Capillary term + Higher order terms

They perform the following steps to convert the Generalized momentum equation to their form.
1. $\Phi_{sf} = 6/5$ in the leading order. For Galilean invariance, they decompose $hU^2 \Phi_{sf} = hU^2 + \dfrac{1}{5} hU^2$. The second term is written as $\dfrac{1}{45} \Lambda^2 h^5$. 
2. They rewrite $\tau_w$ as two terms, one of which is a function of $h_x$ and take it into $P(h)$. What remains is left as $\dfrac{3U}{h} - T$ on the right hand side. 

First, I want to check how accurate step 1 is. 

In [ ]:
xlim1 = 250
xlim2 = 400

fig, ax = plt.subplots(
    figsize=(12, 4),
    constrained_layout=True,
)

ax.plot(
    xc,
    (1/5) * q**2 / h,
    color="tab:blue",
    label=r"Term 1, $\frac{q^2}{5h}$",
)

ax.plot(
    xc,
    (1/45) * Lambda**2 * h**5,
    color="tab:orange",
    label=r"Term 2, $\frac{\Lambda^2 h^5}{45}$",
)

ax.plot(
    xc,
    h,
    color="black",
    label=r"$h$",
)


ax.set_title(r"Comparisons of the flux term replacement")
ax.set_xlabel(r"$x$")
# ax.set_ylabel(r"$$")
ax.set_xlim(xlim1, xlim2)
ax.set_ylim(0, 2.5)
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()

In [ ]:
Lambda

In [ ]:
lam = 100.0       # Increase for stronger smoothing
diff_order = 2

In [ ]:
# ============================================================
# Compute u_xx from the DNS velocity field
# using first_derivative_x() twice
# ============================================================

# Streamwise coordinates
x_1d = x[0, :]

# Allocate arrays
u_x = np.empty_like(u, dtype=float)
u_xx_fd = np.empty_like(u, dtype=float)

# Differentiate each constant-y row in the x direction
for j in range(u.shape[0]):
    u_x[j, :] = first_derivative_x(
        x_1d,
        u[j, :]
    )

    u_xx_fd[j, :] = first_derivative_x(
        x_1d,
        u_x[j, :]
    )

print("u shape:     ", u.shape)
print("u_x shape:   ", u_x.shape)
print("u_xx shape:  ", u_xx_fd.shape)

print(f"u_xx min: {np.nanmin(u_xx_fd):.6e}")
print(f"u_xx max: {np.nanmax(u_xx_fd):.6e}")

In [ ]:
# ============================================================
# Contour plots of u_xx and u
# ============================================================

xlim1 = 360
xlim2 = 365

# Restrict u_xx color scaling to the region being plotted
x_mask = (x[0, :] >= xlim1) & (x[0, :] <= xlim2)

u_xx_plot_region = u_xx_fd[:, x_mask]

# Symmetric color scale for u_xx
vmax = np.nanpercentile(
    np.abs(u_xx_plot_region),
    99
)

levels_uxx = np.linspace(-vmax, vmax, 101)

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 9),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------------------
# Top: u_xx
# ------------------------------------------------------------

cf1 = axes[0].contourf(
    x,
    y,
    u_xx_fd,
    levels=levels_uxx,
    cmap="RdBu_r",
    extend="both"
)

cbar1 = fig.colorbar(cf1, ax=axes[0])
cbar1.set_label(r"$u_{xx}$")

axes[0].set_ylabel(r"$y/h_0$")
axes[0].set_title(
    r"$u_{xx}$ from fourth-order finite differences"
)

axes[0].set_xlim(xlim1, xlim2)

# ------------------------------------------------------------
# Bottom: streamwise velocity u
# ------------------------------------------------------------

u_plot_region = u[:, x_mask]

levels_u = np.linspace(
    np.nanmin(u_plot_region),
    np.nanmax(u_plot_region),
    101
)

cf2 = axes[1].contourf(
    x,
    y,
    u,
    levels=levels_u,
    cmap="viridis",
    extend="both"
)

cbar2 = fig.colorbar(cf2, ax=axes[1])
cbar2.set_label(r"$u$")

axes[1].set_xlabel(r"$x/h_0$")
axes[1].set_ylabel(r"$y/h_0$")
axes[1].set_title(r"Streamwise velocity $u$")

axes[1].set_xlim(xlim1, xlim2)

plt.show()

Directly taking the second derivative of u from DNS can be problematic especially at the interface. The easier way would be to just use the Leibniz integral and calculate the 

In [ ]:
xlim1 = 0
xlim2 = 800

fig, ax = plt.subplots(
    figsize=(12, 4),
    constrained_layout=True,
)

ax.plot(xc, Phi_i, label=r"Interfacial velocity, $u_i$")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$h,\ q,\ u_i$")
ax.set_xlim(xlim1, xlim2)
ax.grid(True)
ax.legend()

plt.show()

In [ ]:
def compute_interface_velocity_liquid_side(
    u,
    f,
    y,
    h,
    n_points=5,
    poly_order=2,
    f_threshold=0.99
):
    """
    Compute liquid-side interfacial velocity u_i(x) by fitting the
    velocity profile in pure-liquid cells immediately below the interface
    and extrapolating the fit to y = h(x).

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Streamwise velocity field.

    f : ndarray, shape (ny, nx)
        Liquid volume fraction field.
        f ~ 1 in liquid and f ~ 0 in gas.

    y : ndarray
        Vertical coordinates. Can be either:
            (ny,)     : 1D vertical coordinate
            (ny, nx)  : 2D coordinate matrix

    h : ndarray, shape (nx,)
        Interface height h(x).

    n_points : int, default=5
        Number of pure-liquid cells closest to the interface used
        for the polynomial fit.

    poly_order : int, default=2
        Polynomial order used for the liquid-side velocity fit.

    f_threshold : float, default=0.99
        Minimum liquid volume fraction for a cell to be considered
        a pure-liquid cell.

    Returns
    -------
    u_i : ndarray, shape (nx,)
        Liquid-side interfacial velocity.

    fit_success : ndarray, shape (nx,), dtype=bool
        True where the requested polynomial fit was successfully performed.
    """

    u = np.asarray(u, dtype=float)
    f = np.asarray(f, dtype=float)
    h = np.asarray(h, dtype=float)

    ny, nx = u.shape

    if f.shape != u.shape:
        raise ValueError("u and f must have the same shape.")

    if h.size != nx:
        raise ValueError(
            f"h has length {h.size}, but u has {nx} x-locations."
        )

    # Handle either 1D or 2D y
    y = np.asarray(y, dtype=float)

    if y.ndim == 1:
        if y.size != ny:
            raise ValueError(
                f"y has length {y.size}, but u has {ny} vertical points."
            )
        y_is_2d = False

    elif y.ndim == 2:
        if y.shape != u.shape:
            raise ValueError(
                "If y is 2D, it must have the same shape as u."
            )
        y_is_2d = True

    else:
        raise ValueError("y must be either 1D or 2D.")

    if n_points < poly_order + 1:
        raise ValueError(
            "n_points must be at least poly_order + 1."
        )

    u_i = np.full(nx, np.nan)
    fit_success = np.zeros(nx, dtype=bool)

    for i in range(nx):

        # Vertical coordinates at this x-location
        if y_is_2d:
            y_col = y[:, i]
        else:
            y_col = y

        # ----------------------------------------------------
        # Find pure-liquid cells below the interface
        # ----------------------------------------------------

        mask = (
            (f[:, i] >= f_threshold)
            & (y_col < h[i])
            & np.isfinite(u[:, i])
            & np.isfinite(y_col)
        )

        indices = np.where(mask)[0]

        if indices.size < poly_order + 1:
            continue

        # ----------------------------------------------------
        # Select cells closest to the interface
        # ----------------------------------------------------

        distance = h[i] - y_col[indices]

        order = np.argsort(distance)

        indices = indices[order[:n_points]]

        y_fit = y_col[indices]
        u_fit = u[indices, i]

        # ----------------------------------------------------
        # Use coordinates relative to interface
        #
        # eta = y - h
        #
        # This improves conditioning and means the polynomial
        # value at eta=0 is directly u_i.
        # ----------------------------------------------------

        eta = y_fit - h[i]

        try:
            coeff = np.polyfit(
                eta,
                u_fit,
                deg=poly_order
            )

            # Evaluate at interface: eta = 0
            u_i[i] = np.polyval(coeff, 0.0)

            fit_success[i] = True

        except (np.linalg.LinAlgError, ValueError):
            continue

    return u_i, fit_success

In [ ]:
u_i, u_i_fit_success = compute_interface_velocity_liquid_side(
    u=u,
    f=f,
    y=y,
    h=h,
    n_points=5,
    poly_order=2,
    f_threshold=0.99
)

print(
    f"Successful fits: "
    f"{np.sum(u_i_fit_success)}/{u_i_fit_success.size}"
)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    xc,
    Phi_i,
    label=r"Original interpolation"
)

plt.plot(
    xc,
    u_i,
    label=r"Liquid-side extrapolation"
)

plt.xlabel(r"$x/h_0$")
plt.ylabel(r"$u_i$")
plt.title(r"Interfacial velocity")
plt.xlim(600, 700)
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# ============================================================
# Diagnose liquid-side extrapolation near the artificial spike
# ============================================================

x_target = 627.3

i = np.argmin(np.abs(xc - x_target))

# Vertical coordinate at this x
if np.asarray(y).ndim == 2:
    y_col = y[:, i]
else:
    y_col = y

u_col = u[:, i]
f_col = f[:, i]

# Pure-liquid cells below interface
mask = (
    (f_col >= 0.99)
    & (y_col < h[i])
    & np.isfinite(u_col)
    & np.isfinite(y_col)
)

indices = np.where(mask)[0]

# Sort according to distance below interface
distance = h[i] - y_col[indices]
order = np.argsort(distance)

# Same number used in extrapolation
selected = indices[order[:5]]

print(f"x = {xc[i]:.6f}")
print(f"h = {h[i]:.6f}")
print(f"Original interpolation = {Phi_i[i]:.6f}")
print(f"Liquid-side extrapolation = {u_i[i]:.6f}")

print("\nSelected points:")
for j in selected:
    print(
        f"y = {y_col[j]:.6f}, "
        f"u = {u_col[j]:.6f}, "
        f"f = {f_col[j]:.6f}, "
        f"h-y = {h[i]-y_col[j]:.6f}"
    )

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 6),
    constrained_layout=True
)

# Entire velocity profile
ax.plot(
    u_col,
    y_col,
    "o-",
    markersize=3,
    label=r"$u(y)$"
)

# Points used by extrapolation
ax.scatter(
    u_col[selected],
    y_col[selected],
    s=80,
    zorder=5,
    label="Points used for fit"
)

# Interface
ax.axhline(
    h[i],
    color="k",
    linestyle="--",
    label=r"$y=h$"
)

# Original interpolation
ax.scatter(
    Phi_i[i],
    h[i],
    s=100,
    marker="s",
    zorder=6,
    label="Original interpolation"
)

# Liquid-side extrapolation
ax.scatter(
    u_i[i],
    h[i],
    s=100,
    marker="x",
    zorder=6,
    label="Liquid-side extrapolation"
)

ax.set_xlabel(r"$u$")
ax.set_ylabel(r"$y$")
ax.set_title(
    rf"Interface velocity extraction at $x/h_0={xc[i]:.2f}$"
)

ax.grid(True)
ax.legend()

plt.show()

Polynomial interpolation on both sides

In [ ]:
def compute_interface_velocity_two_sided(
    u,
    y,
    h,
    n_below=3,
    n_above=3,
    poly_order=2
):
    """
    Estimate u_i(x) = u(x, h(x)) using a local polynomial fit
    through velocity points on BOTH sides of the interface.

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Streamwise velocity field.

    y : ndarray
        Vertical coordinates. Can be:
            (ny,)     : 1D vertical coordinate
            (ny, nx)  : 2D coordinate matrix.

    h : ndarray, shape (nx,)
        Interface height.

    n_below : int
        Number of grid points immediately below the interface.

    n_above : int
        Number of grid points immediately above the interface.

    poly_order : int
        Polynomial order for the local fit.

    Returns
    -------
    u_i : ndarray, shape (nx,)
        Estimated interfacial velocity.

    fit_success : ndarray, shape (nx,), dtype=bool
        Whether the fit was successful.
    """

    u = np.asarray(u, dtype=float)
    y = np.asarray(y, dtype=float)
    h = np.asarray(h, dtype=float)

    ny, nx = u.shape

    if h.size != nx:
        raise ValueError("h must have one value for each x-location.")

    if y.ndim == 1:
        if y.size != ny:
            raise ValueError("y must have ny entries.")
        y_is_2d = False

    elif y.ndim == 2:
        if y.shape != u.shape:
            raise ValueError(
                "If y is 2D, it must have the same shape as u."
            )
        y_is_2d = True

    else:
        raise ValueError("y must be 1D or 2D.")

    n_total = n_below + n_above

    if n_total < poly_order + 1:
        raise ValueError(
            "Need at least poly_order + 1 total points."
        )

    u_i = np.full(nx, np.nan)
    fit_success = np.zeros(nx, dtype=bool)

    for i in range(nx):

        # Vertical coordinates at this x
        if y_is_2d:
            y_col = y[:, i]
        else:
            y_col = y

        u_col = u[:, i]

        valid = (
            np.isfinite(y_col)
            & np.isfinite(u_col)
        )

        # ----------------------------------------------
        # Points below interface
        # ----------------------------------------------

        below = np.where(
            valid & (y_col < h[i])
        )[0]

        if below.size < n_below:
            continue

        distance_below = h[i] - y_col[below]

        below = below[
            np.argsort(distance_below)[:n_below]
        ]

        # ----------------------------------------------
        # Points above interface
        # ----------------------------------------------

        above = np.where(
            valid & (y_col > h[i])
        )[0]

        if above.size < n_above:
            continue

        distance_above = y_col[above] - h[i]

        above = above[
            np.argsort(distance_above)[:n_above]
        ]

        # ----------------------------------------------
        # Combine both sides
        # ----------------------------------------------

        selected = np.concatenate(
            (below, above)
        )

        y_fit = y_col[selected]
        u_fit = u_col[selected]

        # Coordinate relative to interface
        eta = y_fit - h[i]

        try:

            coeff = np.polyfit(
                eta,
                u_fit,
                deg=poly_order
            )

            # Interface corresponds to eta = 0
            u_i[i] = np.polyval(
                coeff,
                0.0
            )

            fit_success[i] = True

        except (np.linalg.LinAlgError, ValueError):
            continue

    return u_i, fit_success

In [ ]:
u_i_twosided, success_twosided = compute_interface_velocity_two_sided(
    u=u,
    y=y,
    h=h,
    n_below=3,
    n_above=3,
    poly_order=2
)

print(
    f"Successful fits: "
    f"{success_twosided.sum()}/{success_twosided.size}"
)

In [ ]:
xlim1 = 0
xlim2 = 800

fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

# Original interpolation
ax.plot(
    xc,
    Phi_i,
    linewidth=1.5,
    label="Original interpolation"
)

# Previous pure-liquid extrapolation
ax.plot(
    xc,
    u_i,
    linewidth=1.2,
    alpha=0.7,
    label="Liquid-side extrapolation"
)

# New two-sided interpolation
ax.plot(
    xc,
    u_i_twosided,
    linewidth=1.5,
    label="Two-sided interpolation"
)

ax.set_xlim(xlim1, xlim2)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_i$")
# ax.set_ylim(0, 1.5)
ax.set_title(
    r"Comparison of interfacial velocity extraction"
)

ax.grid(True)
ax.legend()

plt.show()

Extraction methods are not improving the interfacial velocity. Might as well use SavGol filter later with the raw interpolated quantity

In [ ]:
# ============================================================
# Compute and plot u_i,x
# ============================================================
u_i = Phi_i
u_i_x = first_derivative_x(
    xc,
    u_i
)

xlim1 = 300
xlim2 = 320

fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    u_i_x,
    linewidth=1.2,
    label=r"$u_{i,x}$"
)

ax.axhline(
    0.0,
    color="k",
    linewidth=0.8
)

ax.set_xlim(xlim1, xlim2)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_{i,x}$")
ax.set_title(r"Streamwise derivative of interfacial velocity")

ax.grid(True)
ax.legend()

plt.show()

In [ ]:
m_values = range(21, 42, 2)
p_values = range(2, 4)

best_m_ui, best_p_ui, best_error_ui, ui_sg_results = best_savgol_filter(xc, u_i, m_values=m_values, p_values=p_values, metric="rmse")

print("Best Savitzky-Golay parameters for u_i:")
print(f"  window length m = {best_m_ui}")
print(f"  polynomial order p = {best_p_ui}")
print(f"  RMSE = {best_error_ui:.6e}")

In [ ]:
# ============================================================
# Filter u_i and calculate u_i,x
# ============================================================

dx = np.mean(np.diff(xc))

# Smoothed interfacial velocity
u_i_sg = savgol_filter(
    u_i,
    window_length=best_m_ui,
    polyorder=best_p_ui,
    deriv=0
)

# First derivative directly from Savitzky-Golay polynomial
u_i_x_sg = savgol_filter(
    u_i,
    window_length=best_m_ui,
    polyorder=best_p_ui,
    deriv=1,
    delta=dx
)

In [ ]:
first_derivative_x(xc, u_i_sg)

In [ ]:
xlim1 = 250
xlim2 = 400

fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    u_i,
    linewidth=0.8,
    alpha=0.6,
    label=r"Raw $u_i$"
)

ax.plot(
    xc,
    u_i_sg,
    linewidth=2.0,
    label=rf"SG $u_i$ ($m={best_m_ui}$, $p={best_p_ui}$)"
)

ax.set_xlim(xlim1, xlim2)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_i$")

ax.set_title(
    r"Interfacial velocity: raw vs. Savitzky--Golay"
)

ax.grid(True)
ax.legend()

plt.show()

In [ ]:
xlim1 = 0
xlim2 = 800
fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    u_i_x_sg,
    linewidth=1.5,
    label=rf"SG $u_{{i,x}}$ ($m={best_m_ui}$, $p={best_p_ui}$)"
)

ax.axhline(
    0.0,
    color="k",
    linewidth=0.8
)

ax.set_xlim(xlim1, xlim2)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_{i,x}$")

ax.set_title(
    r"Savitzky--Golay derivative of interfacial velocity"
)

ax.grid(True)
ax.legend()

plt.show()

In [ ]:
def hampel_filter(values, window_size=5, n_sigma=3.0):
    """
    Remove isolated spikes using a Hampel filter.

    Parameters
    ----------
    values : 1D array
        Input signal.

    window_size : int
        Number of points on each side of the center point.

    n_sigma : float
        Outlier threshold in local robust standard deviations.

    Returns
    -------
    filtered : ndarray
        Signal with detected outliers replaced by local median.

    outlier_mask : ndarray of bool
        True where an outlier was detected.
    """

    values = np.asarray(values, dtype=float)

    filtered = values.copy()
    outlier_mask = np.zeros(len(values), dtype=bool)

    # Conversion from MAD to standard deviation
    k = 1.4826

    for i in range(len(values)):

        i0 = max(0, i - window_size)
        i1 = min(len(values), i + window_size + 1)

        local = values[i0:i1]

        median = np.median(local)

        mad = np.median(
            np.abs(local - median)
        )

        sigma = k * mad

        if sigma > 0:
            if np.abs(values[i] - median) > n_sigma * sigma:

                filtered[i] = median
                outlier_mask[i] = True

    return filtered, outlier_mask

In [ ]:
u_i_raw = u_i
u_i_clean, spike_mask = hampel_filter(
    u_i_raw,
    window_size=9,
    n_sigma=2.0
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    u_i_raw,
    linewidth=1.0,
    label=r"Raw $u_i$"
)

ax.scatter(
    xc[spike_mask],
    u_i_raw[spike_mask],
    s=30,
    label="Detected outliers",
    zorder=5
)

ax.plot(
    xc,
    u_i_clean,
    linewidth=1.5,
    label=r"Despiked $u_i$"
)

ax.set_xlim(0, 100)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_i$")

ax.grid(True)
ax.legend()

plt.show()

In [ ]:
# ============================================================
# Detect narrow spikes using local interpolation residual
# ============================================================

u_raw = np.asarray(u_i_raw, dtype=float)

k = 3

residual = np.full_like(u_raw, np.nan)

residual[k:-k] = (
    u_raw[k:-k]
    - 0.5 * (
        u_raw[:-2*k]
        + u_raw[2*k:]
    )
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    residual,
    linewidth=1.0
)

ax.axhline(
    0.0,
    color="k",
    linewidth=0.8
)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$r_i$")

ax.set_title(
    r"Local interpolation residual of $u_i$"
)

ax.grid(True)

plt.show()

In [ ]:
# ============================================================
# Find strongest negative residual events
# ============================================================

from scipy.signal import find_peaks

# Find minima in residual by finding peaks in -residual
peak_indices, properties = find_peaks(
    -residual,
    prominence=0.05,
    distance=10
)

# Sort from strongest negative residual upward
peak_indices = peak_indices[
    np.argsort(residual[peak_indices])
]

# Keep strongest events
n_events = 10
event_indices = peak_indices[:n_events]

print("Strongest negative residual events:")
print()

for rank, i in enumerate(event_indices, start=1):
    print(
        f"{rank:2d}: "
        f"x/h0 = {xc[i]:8.3f}, "
        f"u_i = {u_i_raw[i]:8.4f}, "
        f"residual = {residual[i]: .4e}"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

# ============================================================
# Top: Interfacial velocity
# ============================================================

ax1.plot(
    xc,
    u_i_raw,
    linewidth=1.0,
    label=r"$u_i$"
)

ax1.scatter(
    xc[event_indices],
    u_i_raw[event_indices],
    s=60,
    zorder=5,
    label="Selected events"
)

for rank, i in enumerate(event_indices, start=1):
    ax1.annotate(
        str(rank),
        (xc[i], u_i_raw[i]),
        xytext=(4, 5),
        textcoords="offset points"
    )

ax1.set_ylabel(r"$u_i$")

ax1.set_title(
    r"Strongest narrow events in $u_i$"
)

ax1.grid(True)
ax1.legend()


# ============================================================
# Bottom: Interfacial height
# ============================================================

ax2.plot(
    xc,
    h,
    linewidth=1.0,
    label=r"$h$"
)

# Mark same x-locations as the detected u_i events
ax2.scatter(
    xc[event_indices],
    h[event_indices],
    s=60,
    zorder=5,
    label=r"$u_i$ event locations"
)

# Use same event numbers
for rank, i in enumerate(event_indices, start=1):
    ax2.annotate(
        str(rank),
        (xc[i], h[i]),
        xytext=(4, 5),
        textcoords="offset points"
    )

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$h/h_0$")

ax2.set_title(
    r"Interfacial height at detected $u_i$ events"
)

ax2.grid(True)
ax2.legend()

plt.show()

In [ ]:
for rank, i in enumerate(event_indices[:4], start=1):

    if np.asarray(y).ndim == 2:
        y_col = y[:, i]
    else:
        y_col = y

    u_col = u[:, i]
    f_col = f[:, i]

    fig, ax = plt.subplots(
        figsize=(7, 6),
        constrained_layout=True
    )

    ax.plot(
        u_col,
        y_col,
        "o-",
        markersize=3,
        label=r"$u(y)$"
    )

    ax.axhline(
        h[i],
        linestyle="--",
        label=r"$y=h$"
    )

    ax.scatter(
        u_i_raw[i],
        h[i],
        s=80,
        zorder=5,
        label=r"$u_i$"
    )

    ax.set_xlabel(r"$u$")
    ax.set_ylabel(r"$y$")

    ax.set_title(
        rf"Event {rank}: $x/h_0={xc[i]:.2f}$"
    )

    ax.grid(True)
    ax.legend()

    plt.show()

In [ ]:
# ============================================================
# Plot local streamwise-velocity contours around events
# ============================================================

# Half-width of plotting window in x/h0
window = 3.0

# Which events to inspect
indices_to_plot = event_indices

for rank, i in enumerate(indices_to_plot, start=1):

    x_event = xc[i]

    # --------------------------------------------------------
    # Select local x-window
    # --------------------------------------------------------

    mask_x = (
        (xc >= x_event - window)
        & (xc <= x_event + window)
    )

    idx = np.where(mask_x)[0]

    # --------------------------------------------------------
    # Extract coordinates
    # --------------------------------------------------------

    # x coordinates
    if np.asarray(x).ndim == 2:
        X_local = x[:, idx]
    else:
        X_local = np.tile(
            x[idx][None, :],
            (u.shape[0], 1)
        )

    # y coordinates
    if np.asarray(y).ndim == 2:
        Y_local = y[:, idx]
    else:
        Y_local = np.tile(
            y[:, None],
            (1, len(idx))
        )

    # Velocity
    U_local = u[:, idx]

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 5),
        constrained_layout=True
    )

    contour = ax.contourf(
        X_local,
        Y_local,
        U_local,
        levels=100,
        cmap="RdBu_r"
    )

    cbar = fig.colorbar(
        contour,
        ax=ax
    )

    cbar.set_label(r"$u$")

    # --------------------------------------------------------
    # Plot interface
    # --------------------------------------------------------

    ax.plot(
        xc[idx],
        h[idx],
        "k--",
        linewidth=2.0,
        label=r"$h(x)$"
    )

    # --------------------------------------------------------
    # Mark event location
    # --------------------------------------------------------

    ax.scatter(
        xc[i],
        h[i],
        color="black",
        s=70,
        zorder=10,
        label="Detected event"
    )

    ax.axvline(
        xc[i],
        color="black",
        linestyle=":",
        linewidth=1.0
    )

    # --------------------------------------------------------
    # Zero-velocity contour
    # --------------------------------------------------------

    ax.contour(
        X_local,
        Y_local,
        U_local,
        levels=[0.0],
        colors="black",
        linewidths=1.5
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(r"$x/h_0$")
    ax.set_ylabel(r"$y/h_0$")

    ax.set_title(
        rf"Event {rank}: streamwise velocity around "
        rf"$x/h_0={xc[i]:.2f}$"
    )

    ax.legend()

    plt.show()

In [ ]:
# ============================================================
# Find strongest negative u_i residual events
# and classify them using the corresponding h residual
# ============================================================

import numpy as np
from scipy.signal import find_peaks


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

k = 3
prominence = 0.05
distance = 10
n_events = 20


# ------------------------------------------------------------
# Local interpolation residuals for u_i and h
# ------------------------------------------------------------

u_raw = np.asarray(u_i_raw, dtype=float)
h_raw = np.asarray(h, dtype=float)

residual_u = np.full_like(u_raw, np.nan)
residual_h = np.full_like(h_raw, np.nan)

residual_u[k:-k] = (
    u_raw[k:-k]
    - 0.5 * (
        u_raw[:-2*k]
        + u_raw[2*k:]
    )
)

residual_h[k:-k] = (
    h_raw[k:-k]
    - 0.5 * (
        h_raw[:-2*k]
        + h_raw[2*k:]
    )
)


# ------------------------------------------------------------
# Robust normalization
# ------------------------------------------------------------

valid_u = np.isfinite(residual_u)
valid_h = np.isfinite(residual_h)

median_u = np.median(residual_u[valid_u])
median_h = np.median(residual_h[valid_h])

mad_u = np.median(
    np.abs(residual_u[valid_u] - median_u)
)

mad_h = np.median(
    np.abs(residual_h[valid_h] - median_h)
)

sigma_u = 1.4826 * mad_u
sigma_h = 1.4826 * mad_h

Z_u = residual_u / sigma_u
Z_h = residual_h / sigma_h


# ------------------------------------------------------------
# Find strong negative events in u_i
# ------------------------------------------------------------

peak_indices, properties = find_peaks(
    -residual_u,
    prominence=prominence,
    distance=distance
)

# Sort by strongest negative u_i residual
peak_indices = peak_indices[
    np.argsort(residual_u[peak_indices])
]

# Keep strongest events
event_indices = peak_indices[:n_events]


# ------------------------------------------------------------
# Print both u_i and h behavior
# ------------------------------------------------------------

print("Strongest negative u_i residual events:")
print()

print(
    f"{'Rank':>4s} "
    f"{'x/h0':>10s} "
    f"{'u_i':>10s} "
    f"{'h':>10s} "
    f"{'r_u':>12s} "
    f"{'r_h':>12s} "
    f"{'Z_u':>10s} "
    f"{'Z_h':>10s}"
)

print("-" * 84)

for rank, i in enumerate(event_indices, start=1):

    print(
        f"{rank:4d} "
        f"{xc[i]:10.3f} "
        f"{u_raw[i]:10.4f} "
        f"{h_raw[i]:10.4f} "
        f"{residual_u[i]:12.4e} "
        f"{residual_h[i]:12.4e} "
        f"{Z_u[i]:10.2f} "
        f"{Z_h[i]:10.2f}"
    )

In [ ]:
# ============================================================
# Identify candidate u_i anomalies
# ============================================================

# Threshold for deciding whether h also has significant
# local behavior at the u_i event
Z_h_threshold = 3.0

# Candidate anomaly:
# strong u_i event, but h is locally relatively smooth
anomaly_indices = np.array([
    i for i in event_indices
    if abs(Z_h[i]) < Z_h_threshold
])

print("Candidate u_i anomalies:")
print()

for i in anomaly_indices:

    # Preserve original event number
    rank = np.where(event_indices == i)[0][0] + 1

    print(
        f"Event {rank:2d}: "
        f"x/h0 = {xc[i]:8.3f}, "
        f"u_i = {u_i_raw[i]:8.4f}, "
        f"h = {h[i]:8.4f}, "
        f"Z_u = {Z_u[i]:8.2f}, "
        f"Z_h = {Z_h[i]:8.2f}"
    )

In [ ]:
# ============================================================
# Classify detected u_i events
# ============================================================

Z_h_threshold = 3.0

# u_i event WITHOUT strong corresponding h behavior
anomaly_indices = np.array([
    i for i in event_indices
    if abs(Z_h[i]) < Z_h_threshold
])

# u_i event WITH strong corresponding h behavior
# -> likely physical trough-associated event
ignored_indices = np.array([
    i for i in event_indices
    if abs(Z_h[i]) >= Z_h_threshold
])


# ============================================================
# Plot u_i and h with classifications
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)


# ============================================================
# Top: Interfacial velocity
# ============================================================

ax1.plot(
    xc,
    u_i_raw,
    linewidth=1.0,
    label=r"$u_i$"
)

# Candidate anomalies -- RED
ax1.scatter(
    xc[anomaly_indices],
    u_i_raw[anomaly_indices],
    color="red",
    s=70,
    zorder=6,
    label="Candidate anomalies"
)

# Ignored / trough-associated events -- BLACK
ax1.scatter(
    xc[ignored_indices],
    u_i_raw[ignored_indices],
    color="black",
    s=55,
    zorder=5,
    label="Ignored (associated with $h$)"
)

# Keep original event numbers
for rank, i in enumerate(event_indices, start=1):

    color = "red" if i in anomaly_indices else "black"

    ax1.annotate(
        str(rank),
        (xc[i], u_i_raw[i]),
        xytext=(5, 6),
        textcoords="offset points",
        color=color,
        fontweight="bold"
    )

ax1.set_ylabel(r"$u_i$")

ax1.set_title(
    r"Classification of detected $u_i$ events"
)

ax1.grid(True)
ax1.legend()


# ============================================================
# Bottom: Interfacial height
# ============================================================

ax2.plot(
    xc,
    h,
    linewidth=1.0,
    label=r"$h$"
)

# Same candidate anomalies -- RED
ax2.scatter(
    xc[anomaly_indices],
    h[anomaly_indices],
    color="red",
    s=70,
    zorder=6,
    label="Candidate anomalies"
)

# Same ignored events -- BLACK
ax2.scatter(
    xc[ignored_indices],
    h[ignored_indices],
    color="black",
    s=55,
    zorder=5,
    label="Ignored (associated with $h$)"
)

for rank, i in enumerate(event_indices, start=1):

    color = "red" if i in anomaly_indices else "black"

    ax2.annotate(
        str(rank),
        (xc[i], h[i]),
        xytext=(5, 6),
        textcoords="offset points",
        color=color,
        fontweight="bold"
    )

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$h/h_0$")

ax2.set_title(
    r"Interface geometry at detected $u_i$ events"
)

ax2.grid(True)
ax2.legend()

plt.show()

In [ ]:
# ============================================================
# Find ALL significant negative u_i events
# and retain only candidate u_i anomalies
# ============================================================

import numpy as np
from scipy.signal import find_peaks


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

k = 5

# Minimum prominence required for a negative u_i event
prominence = 0.05

# Minimum spacing between detected events
distance = 10

# Threshold for deciding whether h has significant
# local behavior at the same location
Z_h_threshold = 3.0


# ------------------------------------------------------------
# Input arrays
# ------------------------------------------------------------

u_raw = np.asarray(u_i_raw, dtype=float)
h_raw = np.asarray(h, dtype=float)


# ------------------------------------------------------------
# Local interpolation residuals
#
# r(i) = value(i) - 0.5*[value(i-k) + value(i+k)]
# ------------------------------------------------------------

residual_u = np.full_like(u_raw, np.nan)
residual_h = np.full_like(h_raw, np.nan)

residual_u[k:-k] = (
    u_raw[k:-k]
    - 0.5 * (
        u_raw[:-2*k]
        + u_raw[2*k:]
    )
)

residual_h[k:-k] = (
    h_raw[k:-k]
    - 0.5 * (
        h_raw[:-2*k]
        + h_raw[2*k:]
    )
)


# ------------------------------------------------------------
# Robust normalization of h residual
# ------------------------------------------------------------

valid_h = np.isfinite(residual_h)

median_h = np.median(
    residual_h[valid_h]
)

mad_h = np.median(
    np.abs(
        residual_h[valid_h] - median_h
    )
)

sigma_h = 1.4826 * mad_h

Z_h = residual_h / sigma_h


# ------------------------------------------------------------
# Find ALL significant negative u_i events
# ------------------------------------------------------------

event_indices, properties = find_peaks(
    -residual_u,
    prominence=prominence,
    distance=distance
)


# ------------------------------------------------------------
# Classify events
# ------------------------------------------------------------

# Candidate u_i anomalies:
# u_i has a strong negative event but h does NOT have
# a strong corresponding local feature.
anomaly_indices = np.array([
    i for i in event_indices
    if abs(Z_h[i]) < Z_h_threshold
], dtype=int)


# Ignored events:
# u_i event accompanied by strong local h behavior.
ignored_indices = np.array([
    i for i in event_indices
    if abs(Z_h[i]) >= Z_h_threshold
], dtype=int)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print(f"Total detected u_i events : {len(event_indices)}")
print(f"Candidate u_i anomalies  : {len(anomaly_indices)}")
print(f"Ignored h-associated     : {len(ignored_indices)}")

print("\nCandidate u_i anomalies:")
print()

for i in anomaly_indices:

    print(
        f"x/h0 = {xc[i]:8.3f}, "
        f"u_i = {u_raw[i]:8.4f}, "
        f"h = {h_raw[i]:8.4f}, "
        f"r_u = {residual_u[i]: .4e}, "
        f"r_h = {residual_h[i]: .4e}, "
        f"Z_h = {Z_h[i]:8.2f}"
    )

In [ ]:
# ============================================================
# Plot ALL detected events
#
# RED   = candidate u_i anomaly
# BLACK = ignored because corresponding h behavior exists
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)


# ------------------------------------------------------------
# Top: u_i
# ------------------------------------------------------------

ax1.plot(
    xc,
    u_raw,
    linewidth=1.0,
    label=r"$u_i$"
)

# Candidate anomalies
ax1.scatter(
    xc[anomaly_indices],
    u_raw[anomaly_indices],
    color="red",
    s=50,
    zorder=6,
    label="Candidate anomalies"
)

# Physical / h-associated events
ax1.scatter(
    xc[ignored_indices],
    u_raw[ignored_indices],
    color="black",
    s=40,
    zorder=5,
    label="Ignored ($h$-associated)"
)

ax1.set_ylabel(r"$u_i$")
# ax1.set_xlim(0, 100)
ax1.set_title(
    r"Detected events in interfacial velocity"
)

ax1.grid(True)
ax1.legend()


# ------------------------------------------------------------
# Bottom: h
# ------------------------------------------------------------

ax2.plot(
    xc,
    h_raw,
    linewidth=1.0,
    label=r"$h$"
)

ax2.scatter(
    xc[anomaly_indices],
    h_raw[anomaly_indices],
    color="red",
    s=50,
    zorder=6,
    label="Candidate anomalies"
)

ax2.scatter(
    xc[ignored_indices],
    h_raw[ignored_indices],
    color="black",
    s=40,
    zorder=5,
    label="Ignored ($h$-associated)"
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$h/h_0$")

ax2.set_title(
    r"Interface geometry at detected $u_i$ events"
)

ax2.grid(True)
ax2.legend()

plt.show()

So, for extracting $u_i$, we have explored three options: interpolation, liquid-sided polynomial fit and a two-sided polynomial fit. 

Out of these, 1 and 3 are remarkably similar and I don't see why we use 3 when 1 works as well. 

Liquid-sided polynomial fit is an issue at troughs when h is small leading to just 5 or so grid points covering the liquid part of the domain. Here it overpredicts the values compared to the interpolation and the streamwise evolution looks discontinuous.

Interpolation is our best bet so far but it leads to spurious $u_i$ locations which might be a simulation artifact on the gas side where the velocity looks weird and a little unrealistic. 

To summarize, $u_i$ can be problematic at (i) troughs, and (ii) places with weird gas behavior in short spurts. 

In [ ]:
# ============================================================
# Plot local streamwise-velocity contours around events
# ============================================================

# Half-width of plotting window in x/h0
window = 3.0

# Which events to inspect
indices_to_plot = event_indices

for rank, i in enumerate(indices_to_plot, start=1):

    x_event = xc[i]

    # --------------------------------------------------------
    # Select local x-window
    # --------------------------------------------------------

    mask_x = (
        (xc >= x_event - window)
        & (xc <= x_event + window)
    )

    idx = np.where(mask_x)[0]

    # --------------------------------------------------------
    # Extract coordinates
    # --------------------------------------------------------

    # x coordinates
    if np.asarray(x).ndim == 2:
        X_local = x[:, idx]
    else:
        X_local = np.tile(
            x[idx][None, :],
            (u.shape[0], 1)
        )

    # y coordinates
    if np.asarray(y).ndim == 2:
        Y_local = y[:, idx]
    else:
        Y_local = np.tile(
            y[:, None],
            (1, len(idx))
        )

    # Velocity
    U_local = u[:, idx]

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 5),
        constrained_layout=True
    )

    contour = ax.contourf(
        X_local,
        Y_local,
        U_local,
        levels=100,
        cmap="RdBu_r"
    )

    cbar = fig.colorbar(
        contour,
        ax=ax
    )

    cbar.set_label(r"$u$")

    # --------------------------------------------------------
    # Plot interface
    # --------------------------------------------------------

    ax.plot(
        xc[idx],
        h[idx],
        "k--",
        linewidth=2.0,
        label=r"$h(x)$"
    )

    # --------------------------------------------------------
    # Mark event location
    # --------------------------------------------------------

    ax.scatter(
        xc[i],
        h[i],
        color="black",
        s=70,
        zorder=10,
        label="Detected event"
    )

    ax.axvline(
        xc[i],
        color="black",
        linestyle=":",
        linewidth=1.0
    )

    # --------------------------------------------------------
    # Zero-velocity contour
    # --------------------------------------------------------

    ax.contour(
        X_local,
        Y_local,
        U_local,
        levels=[0.0],
        colors="black",
        linewidths=1.5
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(r"$x/h_0$")
    ax.set_ylabel(r"$y/h_0$")

    ax.set_title(
        rf"Event {rank}: streamwise velocity around "
        rf"$x/h_0={xc[i]:.2f}$"
    )

    ax.legend()
    ax.set_ylim(0, 3)

    plt.show()

In [ ]:
# ============================================================
# Robust normalization of u_i residual
# ============================================================

valid_u = np.isfinite(residual_u)

median_u = np.median(residual_u[valid_u])

mad_u = np.median(
    np.abs(
        residual_u[valid_u] - median_u
    )
)

sigma_u = 1.4826 * mad_u

Z_u = residual_u / sigma_u

In [ ]:
# ============================================================
# Expand each detected anomaly into a contaminated region
# ============================================================

# Neighboring points are considered contaminated while
# their local residual remains larger than this threshold.
Z_u_region_threshold = 2.0

bad_mask = np.zeros(len(u_i_raw), dtype=bool)

for i in anomaly_indices:

    # Always include detected anomaly center
    bad_mask[i] = True

    # --------------------------------------------------------
    # Expand to the left
    # --------------------------------------------------------

    j = i - 1

    while (
        j >= 0
        and np.isfinite(Z_u[j])
        and abs(Z_u[j]) >= Z_u_region_threshold
    ):
        bad_mask[j] = True
        j -= 1

    # --------------------------------------------------------
    # Expand to the right
    # --------------------------------------------------------

    j = i + 1

    while (
        j < len(u_i_raw)
        and np.isfinite(Z_u[j])
        and abs(Z_u[j]) >= Z_u_region_threshold
    ):
        bad_mask[j] = True
        j += 1


bad_indices = np.where(bad_mask)[0]

print(f"Detected anomaly centers : {len(anomaly_indices)}")
print(f"Points marked as bad     : {len(bad_indices)}")

In [ ]:
# ============================================================
# Replace entire contaminated regions
# ============================================================

u_i_corrected = np.asarray(u_i_raw, dtype=float).copy()

good_mask = ~bad_mask

u_i_corrected[bad_mask] = np.interp(
    xc[bad_mask],
    xc[good_mask],
    u_i_corrected[good_mask]
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 5),
    constrained_layout=True
)

ax.plot(
    xc,
    u_i_raw,
    linewidth=1.0,
    label=r"Raw $u_i$"
)

ax.plot(
    xc,
    u_i_corrected,
    linewidth=1.2,
    label=r"Corrected $u_i$"
)

# Entire region being replaced
# ax.scatter(
#     xc[bad_mask],
#     u_i_raw[bad_mask],
#     color="red",
#     s=25,
#     zorder=5,
#     label="Removed region"
# )

# Original detected centers
# ax.scatter(
#     xc[anomaly_indices],
#     u_i_raw[anomaly_indices],
#     color="black",
#     s=60,
#     zorder=6,
#     label="Detected anomaly centers"
# )

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$u_i$")
# ax.set_xlim(0, 100)
ax.grid(True)
ax.legend()

plt.show()

In [ ]:
def compute_ui(
    u,
    y,
    h,
    x,
    *,
    k=3,
    prominence=0.05,
    distance=10,
    Z_h_threshold=3.0,
    Z_u_region_threshold=2.0,
    return_diagnostics=False,
):
    """
    Compute interfacial velocity u_i(x) and repair isolated numerical
    discontinuities that are not associated with physical interface
    features in h(x).

    Procedure
    ---------
    1. Compute raw interfacial velocity by interpolating u(x,y) to y=h(x).

    2. Compute local residuals

           r_u(i) = u_i(i)
                    - 0.5 * [u_i(i-k) + u_i(i+k)]

           r_h(i) = h(i)
                    - 0.5 * [h(i-k) + h(i+k)]

    3. Detect significant negative events in r_u using find_peaks(-r_u).

    4. Use the corresponding normalized h residual to distinguish
       interface-associated events from suspected u_i discontinuities.

       If

           |Z_h| >= Z_h_threshold,

       the event is considered associated with real interface behavior
       and is retained.

       Otherwise, it is flagged as a candidate u_i anomaly.

    5. Expand each candidate anomaly into a contaminated region using
       the normalized u_i residual:

           |Z_u| >= Z_u_region_threshold.

    6. Replace the contaminated region by interpolation between
       neighboring valid u_i values.

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Streamwise velocity field.

    y : ndarray
        Vertical coordinate. Can be either:
            (ny,)
        or
            (ny, nx)

    h : ndarray, shape (nx,)
        Interface height.

    x : ndarray, shape (nx,)
        Streamwise coordinate corresponding to h and u_i.

    k : int, optional
        Number of grid points on either side used to define the
        local residual. Default is 3.

    prominence : float, optional
        Minimum prominence used by find_peaks to identify negative
        u_i residual events. Default is 0.05.

    distance : int, optional
        Minimum grid-point separation between detected events.
        Default is 10.

    Z_h_threshold : float, optional
        Threshold used to determine whether a detected u_i event is
        accompanied by significant local interface behavior.
        Default is 3.0.

    Z_u_region_threshold : float, optional
        Threshold used to expand a detected anomaly into its
        surrounding contaminated region. Default is 2.0.

    return_diagnostics : bool, optional
        If True, also return a dictionary containing the raw u_i,
        residuals, normalized residuals, event indices, anomaly
        centers, ignored events, and complete bad-region mask.

    Returns
    -------
    u_i : ndarray, shape (nx,)
        Corrected interfacial velocity.

    diagnostics : dict, optional
        Returned only when return_diagnostics=True.
    """

    import numpy as np
    from scipy.signal import find_peaks

    # ========================================================
    # Convert inputs
    # ========================================================

    u = np.asarray(u, dtype=float)
    y = np.asarray(y, dtype=float)
    h = np.asarray(h, dtype=float)
    x = np.asarray(x, dtype=float)

    if u.ndim != 2:
        raise ValueError("u must have shape (ny, nx).")

    ny, nx = u.shape

    if h.shape != (nx,):
        raise ValueError(
            f"h must have shape ({nx},), got {h.shape}."
        )

    if x.shape != (nx,):
        raise ValueError(
            f"x must have shape ({nx},), got {x.shape}."
        )

    if y.ndim == 1:
        if y.shape != (ny,):
            raise ValueError(
                f"1D y must have shape ({ny},), got {y.shape}."
            )

    elif y.ndim == 2:
        if y.shape != u.shape:
            raise ValueError(
                "2D y must have the same shape as u."
            )

    else:
        raise ValueError(
            "y must be either 1D or 2D."
        )

    if k < 1:
        raise ValueError("k must be >= 1.")

    if nx <= 2 * k:
        raise ValueError(
            "Not enough x points for the requested k."
        )

    # ========================================================
    # 1. Raw interfacial velocity
    # ========================================================

    u_i_raw = np.full(nx, np.nan)

    for i in range(nx):

        if y.ndim == 1:
            y_col = y
        else:
            y_col = y[:, i]

        u_col = u[:, i]

        valid = (
            np.isfinite(y_col)
            & np.isfinite(u_col)
        )

        if np.count_nonzero(valid) < 2:
            continue

        y_valid = y_col[valid]
        u_valid = u_col[valid]

        # Sort in y in case the grid is not strictly ordered
        order = np.argsort(y_valid)

        y_valid = y_valid[order]
        u_valid = u_valid[order]

        u_i_raw[i] = np.interp(
            h[i],
            y_valid,
            u_valid
        )

    # ========================================================
    # 2. Local residuals
    # ========================================================

    residual_u = np.full(nx, np.nan)
    residual_h = np.full(nx, np.nan)

    residual_u[k:-k] = (
        u_i_raw[k:-k]
        - 0.5 * (
            u_i_raw[:-2*k]
            + u_i_raw[2*k:]
        )
    )

    residual_h[k:-k] = (
        h[k:-k]
        - 0.5 * (
            h[:-2*k]
            + h[2*k:]
        )
    )

    # ========================================================
    # 3. Robust residual scales
    # ========================================================

    def robust_scale(values):

        valid = np.isfinite(values)

        if not np.any(valid):
            return np.nan

        values_valid = values[valid]

        median = np.median(values_valid)

        mad = np.median(
            np.abs(values_valid - median)
        )

        return 1.4826 * mad

    sigma_u = robust_scale(residual_u)
    sigma_h = robust_scale(residual_h)

    if not np.isfinite(sigma_u) or sigma_u <= 0:
        raise ValueError(
            "Unable to determine a non-zero robust scale "
            "for the u_i residual."
        )

    if not np.isfinite(sigma_h) or sigma_h <= 0:
        raise ValueError(
            "Unable to determine a non-zero robust scale "
            "for the h residual."
        )

    Z_u = residual_u / sigma_u
    Z_h = residual_h / sigma_h

    # ========================================================
    # 4. Detect ALL significant negative u_i events
    # ========================================================

    residual_for_peaks = residual_u.copy()

    # find_peaks does not handle NaNs reliably.
    residual_for_peaks[
        ~np.isfinite(residual_for_peaks)
    ] = 0.0

    event_indices, properties = find_peaks(
        -residual_for_peaks,
        prominence=prominence,
        distance=distance
    )

    # ========================================================
    # 5. Separate physical/interface-associated events from
    #    candidate u_i anomalies
    # ========================================================

    anomaly_indices = np.array(
        [
            i for i in event_indices
            if (
                np.isfinite(Z_h[i])
                and abs(Z_h[i]) < Z_h_threshold
            )
        ],
        dtype=int
    )

    ignored_indices = np.array(
        [
            i for i in event_indices
            if (
                np.isfinite(Z_h[i])
                and abs(Z_h[i]) >= Z_h_threshold
            )
        ],
        dtype=int
    )

    # ========================================================
    # 6. Expand each anomaly into its contaminated neighborhood
    # ========================================================

    bad_mask = np.zeros(nx, dtype=bool)

    for i in anomaly_indices:

        # Anomaly center is always bad
        bad_mask[i] = True

        # ----------------------------------------------------
        # Expand left
        # ----------------------------------------------------

        j = i - 1

        while (
            j >= 0
            and np.isfinite(Z_u[j])
            and abs(Z_u[j]) >= Z_u_region_threshold
        ):
            bad_mask[j] = True
            j -= 1

        # ----------------------------------------------------
        # Expand right
        # ----------------------------------------------------

        j = i + 1

        while (
            j < nx
            and np.isfinite(Z_u[j])
            and abs(Z_u[j]) >= Z_u_region_threshold
        ):
            bad_mask[j] = True
            j += 1

    bad_indices = np.where(bad_mask)[0]

    # ========================================================
    # 7. Replace contaminated regions
    # ========================================================

    u_i_corrected = u_i_raw.copy()

    if np.any(bad_mask):

        good_mask = (
            ~bad_mask
            & np.isfinite(u_i_raw)
        )

        if np.count_nonzero(good_mask) < 2:
            raise ValueError(
                "Not enough valid u_i values remain to "
                "interpolate across anomalous regions."
            )

        u_i_corrected[bad_mask] = np.interp(
            x[bad_mask],
            x[good_mask],
            u_i_raw[good_mask]
        )

    # ========================================================
    # 8. Diagnostics
    # ========================================================

    diagnostics = {
        "u_i_raw": u_i_raw,
        "u_i_corrected": u_i_corrected,

        "residual_u": residual_u,
        "residual_h": residual_h,

        "Z_u": Z_u,
        "Z_h": Z_h,

        "sigma_u": sigma_u,
        "sigma_h": sigma_h,

        "event_indices": event_indices,
        "anomaly_indices": anomaly_indices,
        "ignored_indices": ignored_indices,

        "bad_mask": bad_mask,
        "bad_indices": bad_indices,

        "peak_properties": properties,
    }

    if return_diagnostics:
        return u_i_corrected, diagnostics

    return u_i_corrected

In [ ]:
# u:
#   Streamwise velocity field with shape (ny, nx).
#   u[:, i] gives the vertical velocity profile at x[i].

# y:
#   Vertical coordinate used to interpolate u(x,y) to the interface y = h(x).
#   Can have shape (ny,) or (ny, nx).

# h:
#   Interfacial height h(x), shape (nx,).
#   Also used to determine whether an unusual u_i value is associated with
#   a physical interface feature such as a trough.

# xc:
#   Streamwise coordinate, shape (nx,).
#   Must correspond to the x-locations of h and the columns of u.

# k:
#   Half-width, in x-grid points, used to define the local residual:
#
#       r_u[i] = u_i[i] - 0.5*(u_i[i-k] + u_i[i+k])
#
#   with the same definition applied to h.
#
#   Increasing k:
#       - compares each point against a wider neighborhood
#       - detects broader departures from local behavior
#       - may become more sensitive to genuine physical variations
#
#   Decreasing k:
#       - makes the comparison more local
#       - is better targeted at very narrow discontinuities
#       - may not identify the full behavior of wider anomalous regions

# prominence:
#   Minimum prominence of a negative u_i residual required for it to be
#   identified as an event by find_peaks().
#
#   Increasing prominence:
#       - detects fewer events
#       - retains only stronger/more obvious discontinuities
#
#   Decreasing prominence:
#       - detects weaker events
#       - increases sensitivity, but may also detect normal fluctuations
#
#   This is expressed in the units of u_i because find_peaks() is applied
#   to the unnormalized residual.

# distance:
#   Minimum separation, in x-grid points, between detected events.
#
#   Increasing distance:
#       - prevents several nearby residual minima from being identified
#         as separate events
#       - useful when one bad region generates multiple local minima
#
#   Decreasing distance:
#       - allows distinct nearby events to be detected separately

# Z_h_threshold:
#   Determines whether a detected u_i event is accompanied by sufficiently
#   unusual local behavior in h.
#
#       |Z_h| >= Z_h_threshold
#           -> h is also locally unusual
#           -> event is treated as potentially physical and is NOT repaired
#
#       |Z_h| < Z_h_threshold
#           -> h does not show a corresponding local feature
#           -> event is classified as a candidate u_i anomaly
#
#   Increasing Z_h_threshold:
#       - makes it harder for h to explain a u_i event
#       - MORE detected events will be classified as u_i anomalies
#
#   Decreasing Z_h_threshold:
#       - makes it easier for h to explain a u_i event
#       - FEWER detected events will be classified as anomalies

# Z_u_region_threshold:
#   Once an anomaly center has been identified, this controls how far the
#   contaminated region is expanded around that center.
#
#   Neighboring points are included in the bad region while:
#
#       |Z_u| >= Z_u_region_threshold
#
#   Increasing Z_u_region_threshold:
#       - produces narrower corrected regions
#       - preserves more of the original u_i data
#       - risks leaving contaminated neighboring points
#
#   Decreasing Z_u_region_threshold:
#       - expands the corrected region farther from the anomaly center
#       - removes more of the potentially contaminated neighborhood
#       - risks removing legitimate physical variation if set too low
#
#   Therefore:
#       prominence            -> determines whether an event is detected
#       Z_h_threshold         -> determines whether that event is considered
#                                physical or anomalous
#       Z_u_region_threshold  -> determines the spatial extent of the repair

# return_diagnostics:
#   If True, return both the corrected u_i and a dictionary containing the
#   raw u_i, residuals, Z-scores, detected events, anomaly centers,
#   ignored physical events, and complete bad-region mask.
#
#   If False, return only the corrected u_i.


u_i, ui_diag = compute_ui(
    u,
    y,
    h,
    xc,
    k=5,
    prominence=0.05,
    distance=10,
    Z_h_threshold=3.0,
    Z_u_region_threshold=2.0,
    return_diagnostics=True
)

In [ ]:
# ============================================================
# Compare old Phi_i with new corrected u_i
# ============================================================

Phi_i_arr = np.asarray(Phi_i, dtype=float)
u_i_arr = np.asarray(u_i, dtype=float)

difference = u_i_arr - Phi_i_arr
bad_mask = ui_diag["bad_mask"]


fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 11),
    sharex=True,
    constrained_layout=True
)


# ============================================================
# Top: Direct comparison
# ============================================================

ax1.plot(
    xc,
    Phi_i_arr,
    linewidth=1.0,
    label=r"Original $\Phi_i$"
)

ax1.plot(
    xc,
    u_i_arr,
    linewidth=1.2,
    label=r"New corrected $u_i$"
)

ax1.scatter(
    xc[bad_mask],
    u_i_arr[bad_mask],
    color="red",
    s=20,
    zorder=5,
    label="Corrected region"
)

ax1.set_ylabel(r"Interfacial velocity")

ax1.set_title(
    r"Comparison of original $\Phi_i$ and corrected $u_i$"
)

ax1.grid(True)
ax1.legend()


# ============================================================
# Middle: Difference
# ============================================================

ax2.plot(
    xc,
    difference,
    linewidth=1.0
)

ax2.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=1.0
)

ax2.scatter(
    xc[bad_mask],
    difference[bad_mask],
    color="red",
    s=20,
    zorder=5
)

ax2.set_ylabel(
    r"$u_i-\Phi_i$"
)

ax2.set_title(
    r"Change introduced by the new $u_i$ reconstruction"
)

ax2.grid(True)


# ============================================================
# Bottom: Corrected u_i only
# ============================================================

ax3.plot(
    xc,
    u_i_arr,
    linewidth=1.0,
    label=r"$u_i$"
)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel(r"$u_i$")

ax3.set_title(
    r"Corrected interfacial velocity $u_i$"
)

ax3.grid(True)
ax3.legend()


plt.show()

In [ ]:
# ============================================================
# Compute x-derivative of corrected interfacial velocity
# ============================================================

u_i_x = first_derivative_x(
    xc,
    u_i
)


# ============================================================
# Plot u_i and du_i/dx
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)


# ------------------------------------------------------------
# Top: Corrected interfacial velocity
# ------------------------------------------------------------

ax1.plot(
    xc,
    u_i,
    linewidth=1.0,
    label=r"$u_i$"
)

ax1.set_ylabel(r"$u_i$")

ax1.set_title(
    r"Corrected interfacial velocity"
)

ax1.grid(True)
ax1.legend()


# ------------------------------------------------------------
# Bottom: x-derivative of interfacial velocity
# ------------------------------------------------------------

ax2.plot(
    xc,
    u_i_x,
    linewidth=1.0,
    label=r"$\partial u_i/\partial x$"
)

ax2.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(
    r"$\partial u_i/\partial x$"
)

ax2.set_title(
    r"Streamwise derivative of interfacial velocity"
)

ax2.grid(True)
ax2.legend()
ax2.set_xlim(220,240)
plt.show()

In [ ]:
m_values = range(11, 42, 2)
p_values = range(2, 4)

best_m_ui, best_p_ui, best_error_ui, results_ui = best_savgol_filter(
    xc,
    u_i,
    m_values,
    p_values,
    metric="rmse",
    power=4
)

print(
    f"u_i: Best m={best_m_ui}, "
    f"p={best_p_ui}, "
    f"RMSE={best_error_ui:.6e}"
)


In [ ]:
u_i_filtered = savgol_filter(
    u_i,
    window_length=best_m_ui,
    polyorder=best_p_ui
)


# ============================================================
# Compute spatial derivative from filtered u_i
# ============================================================

# u_i_x = first_derivative_x(
#     xc,
#     u_i_filtered
# )


# ============================================================
# Find best Savitzky-Golay filter for du_i/dx
# ============================================================

best_m_uix, best_p_uix, best_error_uix, results_uix = best_savgol_filter(
    xc,
    u_i_x,
    m_values,
    p_values,
    metric="rmse",
    power=4
)

print(
    f"du_i/dx: Best m={best_m_uix}, "
    f"p={best_p_uix}, "
    f"RMSE={best_error_uix:.6e}"
)


# ============================================================
# Apply optimal filter to du_i/dx
# ============================================================

u_i_x_filtered = savgol_filter(
    u_i_x,
    window_length=best_m_uix,
    polyorder=best_p_uix
)


# ============================================================
# Plot original and filtered quantities
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)


# ------------------------------------------------------------
# u_i
# ------------------------------------------------------------

ax1.plot(
    xc,
    u_i,
    linewidth=1.0,
    alpha=0.7,
    label=r"Original $u_i$"
)

ax1.plot(
    xc,
    u_i_filtered,
    linewidth=1.3,
    label=(
        rf"Filtered $u_i$ "
        rf"($m={best_m_ui}$, $p={best_p_ui}$)"
    )
)

ax1.set_ylabel(r"$u_i$")

ax1.set_title(
    r"Original and Savitzky-Golay filtered interfacial velocity"
)

ax1.grid(True)
ax1.legend()


# ------------------------------------------------------------
# du_i/dx
# ------------------------------------------------------------

ax2.plot(
    xc,
    u_i_x,
    linewidth=2.0,
    alpha=0.7,
    label=r"Original $\partial u_i/\partial x$"
)

ax2.plot(
    xc,
    u_i_x_filtered,
    linewidth=1.3,
    label=(
        rf"Filtered $\partial u_i/\partial x$ "
        rf"($m={best_m_uix}$, $p={best_p_uix}$)"
    )
)

ax2.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(
    r"$\partial u_i/\partial x$"
)

ax2.set_title(
    r"Original and Savitzky-Golay filtered interfacial-velocity derivative"
)

ax2.grid(True)
ax2.legend()
ax2.set_xlim(300,400)
plt.show()

Really happy with this workflow for u_i and u_i_x.

In [ ]:
u_i = u_i_filtered
u_i_x = u_i_x_filtered

In [ ]:
# ============================================================
# Streamwise viscous diffusion contributions
# ============================================================

D_qxx = q_xx

D_ui_hxx = -u_i * h_xx

D_hx_uix = -2.0 * h_x * u_i_x


# Total contribution
D_total = (
    D_qxx
    + D_ui_hxx
    + D_hx_uix
) / Re


# Contribution excluding q_xx
D_other = (
    D_ui_hxx
    + D_hx_uix
) / Re


# ============================================================
# Plot
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)


# ------------------------------------------------------------
# Top: Individual contributions
# ------------------------------------------------------------

ax1.plot(
    xc,
    D_qxx / Re,
    linewidth=1.0,
    label=r"$q_{xx}/Re$"
)

ax1.plot(
    xc,
    D_ui_hxx / Re,
    linewidth=1.0,
    label=r"$-u_i h_{xx}/Re$"
)

ax1.plot(
    xc,
    D_hx_uix / Re,
    linewidth=1.0,
    label=r"$-2h_x u_{i,x}/Re$"
)

ax1.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax1.set_ylabel("Contribution")
ax1.set_title("Components of streamwise viscous diffusion")
ax1.grid(True)
ax1.legend()


# ------------------------------------------------------------
# Bottom:
# Total contribution vs contribution excluding q_xx
# ------------------------------------------------------------

ax2.plot(
    xc,
    D_total,
    linewidth=1.2,
    label=r"$D_{\mathrm{total}} = "
          r"(q_{xx}-u_i h_{xx}-2h_xu_{i,x})/Re$"
)

ax2.plot(
    xc,
    D_other,
    linewidth=1.2,
    label=r"$D_{\mathrm{other}} = "
          r"(-u_i h_{xx}-2h_xu_{i,x})/Re$"
)

ax2.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel("Contribution")

ax2.set_title(
    r"Total streamwise viscous diffusion and contribution excluding $q_{xx}$"
)

ax2.grid(True)
ax2.legend()
ax2.set_xlim(300,400)
plt.show()

In [ ]:
def compute_interface_uz(
    u,
    y,
    h,
    f=None,
    f_threshold=0.99,
    n_points=5,
    polyorder=2,
):
    """
    Compute the liquid-side vertical velocity gradient

        u_z|_h = (∂u/∂z)|_{z=h^-}

    using a local polynomial fit to liquid-side velocity values
    immediately below the interface.

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Streamwise DNS velocity field.

    y : ndarray
        Vertical coordinate. Can have shape (ny,) or (ny, nx).

    h : ndarray, shape (nx,)
        Interface height.

    f : ndarray, shape (ny, nx), optional
        VOF liquid fraction. If supplied, only cells satisfying
        f >= f_threshold are used.

    f_threshold : float, default=0.99
        Minimum liquid fraction for a point to be considered
        a pure-liquid point.

    n_points : int, default=5
        Number of liquid-side points closest to the interface
        used in the polynomial fit.

    polyorder : int, default=2
        Polynomial order used for the local fit.

    Returns
    -------
    u_z_i : ndarray, shape (nx,)
        Liquid-side estimate of (∂u/∂z)|_h.
    """

    u = np.asarray(u, dtype=float)
    y = np.asarray(y, dtype=float)
    h = np.asarray(h, dtype=float)

    ny, nx = u.shape

    if h.shape != (nx,):
        raise ValueError("h must have shape (nx,).")

    if polyorder >= n_points:
        raise ValueError("polyorder must be smaller than n_points.")

    if f is not None:
        f = np.asarray(f, dtype=float)

        if f.shape != u.shape:
            raise ValueError("f must have the same shape as u.")

    u_z_i = np.full(nx, np.nan)


    for i in range(nx):

        # Vertical coordinates for this x-column
        if y.ndim == 1:
            y_col = y
        else:
            y_col = y[:, i]

        u_col = u[:, i]


        # ----------------------------------------------------
        # Select liquid-side points below the interface
        # ----------------------------------------------------

        mask = y_col < h[i]

        if f is not None:
            mask &= (f[:, i] >= f_threshold)

        indices = np.where(mask)[0]

        if len(indices) < n_points:
            continue


        # Closest liquid points to interface
        distances = h[i] - y_col[indices]

        order = np.argsort(distances)

        indices = indices[order[:n_points]]


        # ----------------------------------------------------
        # Local coordinate relative to interface
        #
        # eta = z - h
        #
        # Interface therefore corresponds exactly to eta = 0.
        # ----------------------------------------------------

        eta = y_col[indices] - h[i]
        u_fit = u_col[indices]


        # Sort for consistency
        sort_idx = np.argsort(eta)

        eta = eta[sort_idx]
        u_fit = u_fit[sort_idx]


        # ----------------------------------------------------
        # Polynomial fit
        # ----------------------------------------------------

        coeff = np.polyfit(
            eta,
            u_fit,
            polyorder
        )

        # Differentiate polynomial
        dcoeff = np.polyder(coeff)

        # Evaluate derivative at interface: eta = 0
        u_z_i[i] = np.polyval(
            dcoeff,
            0.0
        )


    return u_z_i

In [ ]:
def compute_interface_uz_from_ui(
    u,
    y,
    h,
    u_i,
    n_cfd_points=3
):
    """
    Compute liquid-side du/dz at the interface using:

        1 artificial point: (h, u_i)
        + n_cfd_points actual CFD points immediately below h.

    For n_cfd_points=3, a cubic polynomial is constructed through
    the four points and differentiated exactly at z=h.

    Parameters
    ----------
    u : ndarray, shape (ny, nx)
        Streamwise velocity field.

    y : ndarray
        Vertical coordinates, either:
            (ny,)
        or
            (ny, nx)

    h : ndarray, shape (nx,)
        Interface height.

    u_i : ndarray, shape (nx,)
        Previously reconstructed/filtered interfacial velocity.

    n_cfd_points : int, default=3
        Number of actual CFD points immediately below h.

    Returns
    -------
    u_z_i : ndarray, shape (nx,)
        Estimate of du/dz at z=h from the liquid side.
    """

    u = np.asarray(u, dtype=float)
    y = np.asarray(y, dtype=float)
    h = np.asarray(h, dtype=float)
    u_i = np.asarray(u_i, dtype=float)

    ny, nx = u.shape

    if h.shape != (nx,):
        raise ValueError("h must have shape (nx,).")

    if u_i.shape != (nx,):
        raise ValueError("u_i must have shape (nx,).")

    u_z_i = np.full(nx, np.nan)


    for i in range(nx):

        # -----------------------------------------------
        # Vertical coordinate for this x-column
        # -----------------------------------------------

        if y.ndim == 1:
            y_col = y
        elif y.ndim == 2:
            y_col = y[:, i]
        else:
            raise ValueError("y must be 1D or 2D.")

        u_col = u[:, i]


        # -----------------------------------------------
        # Find CFD grid points below the interface
        # -----------------------------------------------

        below = np.where(y_col < h[i])[0]

        if len(below) < n_cfd_points:
            continue


        # Sort according to distance from interface
        distance = h[i] - y_col[below]

        order = np.argsort(distance)

        idx = below[order[:n_cfd_points]]


        # -----------------------------------------------
        # Construct local stencil
        #
        # First point is artificial interface point.
        # Remaining points are actual CFD data.
        # -----------------------------------------------

        z_stencil = np.concatenate((
            [h[i]],
            y_col[idx]
        ))

        u_stencil = np.concatenate((
            [u_i[i]],
            u_col[idx]
        ))


        # -----------------------------------------------
        # Shift coordinates so interface is eta = 0.
        #
        # This substantially improves conditioning.
        # -----------------------------------------------

        eta = z_stencil - h[i]


        # -----------------------------------------------
        # Polynomial through all stencil points
        #
        # 4 points -> cubic polynomial
        # -----------------------------------------------

        degree = len(eta) - 1

        coeff = np.polyfit(
            eta,
            u_stencil,
            degree
        )


        # -----------------------------------------------
        # Differentiate polynomial
        # -----------------------------------------------

        dcoeff = np.polyder(coeff)


        # -----------------------------------------------
        # Evaluate derivative at interface:
        #
        # eta = 0  <=>  z = h
        # -----------------------------------------------

        u_z_i[i] = np.polyval(
            dcoeff,
            0.0
        )


    return u_z_i

In [ ]:
u_z_i = compute_interface_uz_from_ui(
    u,
    y,
    h,
    u_i,
    n_cfd_points=4
)

In [ ]:
D_shear = h_x**2 * u_z_i

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 7),
    sharex=True,
    constrained_layout=True
)

ax1.plot(
    xc,
    u_z_i,
    linewidth=1.0
)

ax1.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax1.set_ylabel(r"$u_z|_{h^-}$")
ax1.set_title("Liquid-side interfacial velocity gradient")
ax1.grid(True)


ax2.plot(
    xc,
    D_shear / Re,
    linewidth=1.0
)

ax2.axhline(
    0.0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$h_x^2 u_z|_{h^-}/Re$")
ax2.set_title(
    r"Interfacial-gradient contribution to $\int_0^h u_{xx}\,dz/Re$"
)
ax2.set_xlim(0, 800)
ax2.set_ylim(-0.25,0.25)
ax2.grid(True)

plt.show()

I'm gonna ignore the final term.

Pressure now

In [ ]:
def compute_px(x, p):
    """
    Compute p_x = dp/dx at fixed y.

    Parameters
    ----------
    x : array
        Streamwise coordinate. Can be:
            (nx,) or
            (ny, nx)

    p : array, shape (ny, nx)
        DNS pressure field.

    Returns
    -------
    p_x : array, shape (ny, nx)
        Streamwise pressure gradient at fixed y.
    """

    x = np.asarray(x, dtype=float)
    p = np.asarray(p, dtype=float)

    if p.ndim != 2:
        raise ValueError("p must have shape (ny, nx).")

    ny, nx = p.shape

    if x.ndim == 1:
        if x.shape != (nx,):
            raise ValueError("1D x must have shape (nx,).")
        x_1d = x

    elif x.ndim == 2:
        if x.shape != p.shape:
            raise ValueError("2D x must have the same shape as p.")
        x_1d = x[0, :]

    else:
        raise ValueError("x must be 1D or 2D.")

    p_x = np.empty_like(p)

    # Differentiate along x at every fixed-y location
    for j in range(ny):
        p_x[j, :] = first_derivative_x(
            x_1d,
            p[j, :]
        )

    return p_x

In [ ]:
p_x = compute_px(xc, p)

In [ ]:
# Construct 2D x-coordinate array matching y and p_x
X = np.broadcast_to(xc[None, :], p_x.shape)

print("X shape:  ", X.shape)
print("y shape:  ", y.shape)
print("p_x shape:", p_x.shape)

In [ ]:
# Liquid mask
liquid_mask = (f >= 0.5)

px_liquid_values = p_x[liquid_mask]

print("Liquid p_x statistics:")
print(f"min       = {np.nanmin(px_liquid_values):.6e}")
print(f"max       = {np.nanmax(px_liquid_values):.6e}")
print(f"mean      = {np.nanmean(px_liquid_values):.6e}")
print(f"median    = {np.nanmedian(px_liquid_values):.6e}")

for percentile in [1, 5, 25, 50, 75, 95, 99]:
    print(
        f"{percentile:2d}th percentile = "
        f"{np.nanpercentile(px_liquid_values, percentile):.6e}"
    )

In [ ]:
# Mask gas
p_x_liquid = np.ma.masked_where(f < 0.5, p_x)

vlim = 0.6

fig, ax = plt.subplots(figsize=(14, 5))

levels = np.linspace(-vlim, vlim, 121)

cf = ax.contourf(
    X,
    y,
    p_x_liquid,
    levels=levels,
    cmap="RdBu_r",
    extend="both"
)

ax.plot(
    xc,
    h,
    "k--",
    linewidth=1.2,
    label=r"$h(x)$"
)

cbar = plt.colorbar(cf, ax=ax)
cbar.set_label(r"$\partial p/\partial x$")

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(r"$y/h_0$")
ax.set_ylim(0, 1.05*np.nanmax(h))

ax.set_title(
    r"Liquid-phase DNS pressure gradient $\partial p/\partial x$"
)

ax.legend()
ax.set_xlim(300,350)
plt.show()

In [ ]:
def integrate_px_to_fraction(y, p_x, h, fractions=(0.90, 0.95, 0.98, 1.00)):
    """
    Integrate p_x vertically from y=0 to y=alpha*h(x).

    Returns
    -------
    integrals : dict
        integrals[alpha] = integral_0^(alpha*h) p_x dy
    """

    nx = p_x.shape[1]

    integrals = {
        alpha: np.full(nx, np.nan)
        for alpha in fractions
    }

    for i in range(nx):

        y_col = y[:, i] if y.ndim == 2 else y
        px_col = p_x[:, i]

        for alpha in fractions:

            y_top = alpha * h[i]

            # Points strictly below integration limit
            mask = y_col < y_top

            y_use = y_col[mask]
            px_use = px_col[mask]

            if len(y_use) < 2:
                continue

            # Interpolate p_x to exact upper integration limit
            px_top = np.interp(
                y_top,
                y_col,
                px_col
            )

            # Add exact upper point
            y_int = np.append(y_use, y_top)
            px_int = np.append(px_use, px_top)

            integrals[alpha][i] = np.trapezoid(
                px_int,
                y_int
            )

    return integrals

In [ ]:
Ip_fraction = integrate_px_to_fraction(
    y,
    p_x,
    h,
    fractions=(0.90, 0.95, 0.98, 1.00)
)

fig, ax = plt.subplots(figsize=(14, 5))

for alpha in (0.90, 0.95):

    ax.plot(
        xc,
        Ip_fraction[alpha],
        linewidth=1.0,
        label=rf"$\alpha={alpha:.2f}$"
    )

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax.set_xlabel(r"$x/h_0$")
ax.set_ylabel(
    r"$\int_0^{\alpha h} p_x\,dy$"
)

ax.set_title(
    "Sensitivity of integrated pressure gradient "
    "to the near-interface region"
)

ax.grid(True)
ax.legend()
ax.set_xlim(300,350)
ax.set_ylim(-1,1)
plt.show()

I am pretty happy with what we have for curvature. I'm going to keep using the filtered one instead of the raw quantity. Now what I am thinking of is how to extract the remaining pressure term and what exactly is important here

In [ ]:
def compute_interface_pressure(p, y, h):
    """
    Linearly interpolate the DNS pressure field to the interface z = h(x).

    Parameters
    ----------
    p : ndarray, shape (ny, nx)
        DNS pressure field.

    y : ndarray, shape (ny, nx) or (ny,)
        Vertical coordinates.

    h : ndarray, shape (nx,)
        Interface height.

    Returns
    -------
    p_i : ndarray, shape (nx,)
        Pressure interpolated to z = h(x).
    """

    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=float)
    h = np.asarray(h, dtype=float)

    ny, nx = p.shape

    if h.shape != (nx,):
        raise ValueError(
            f"h must have shape ({nx},), got {h.shape}."
        )

    p_i = np.full(nx, np.nan)

    for i in range(nx):

        # Vertical coordinates at this x location
        if y.ndim == 1:
            y_col = y
        elif y.ndim == 2:
            y_col = y[:, i]
        else:
            raise ValueError("y must be either 1D or 2D.")

        p_col = p[:, i]

        # Remove invalid values
        valid = np.isfinite(y_col) & np.isfinite(p_col)

        y_use = y_col[valid]
        p_use = p_col[valid]

        if len(y_use) < 2:
            continue

        # Make sure y is monotonically increasing
        order = np.argsort(y_use)

        y_use = y_use[order]
        p_use = p_use[order]

        # Only interpolate -- do not extrapolate
        if h[i] < y_use[0] or h[i] > y_use[-1]:
            continue

        p_i[i] = np.interp(
            h[i],
            y_use,
            p_use
        )

    return p_i

In [ ]:
p_i = compute_interface_pressure(
    p,
    y,
    h
)

In [ ]:
anomaly_indices = ui_diag["anomaly_indices"]
bad_mask = ui_diag["bad_mask"]

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

# ---------------------------------------------------------
# Interface height
# ---------------------------------------------------------

ax1.plot(
    xc,
    h,
    linewidth=1.2,
    label=r"$h$"
)

# Highlight locations where u_i was corrected
ax1.scatter(
    xc[bad_mask],
    h[bad_mask],
    s=35,
    marker="o",
    color="red",
    zorder=5,
    label=r"$u_i$ corrected region"
)

ax1.set_ylabel(r"$h/h_0$")
ax1.set_title(
    r"Interface height and interfacial pressure at $u_i$ anomaly locations"
)

ax1.grid(True)
ax1.legend()


# ---------------------------------------------------------
# Interfacial pressure
# ---------------------------------------------------------

ax2.plot(
    xc,
    p_i,
    linewidth=1.2,
    label=r"$p_i$"
)

# Highlight locations where u_i was corrected
ax2.scatter(
    xc[bad_mask],
    p_i[bad_mask],
    s=35,
    marker="o",
    color="red",
    zorder=5,
    label=r"$u_i$ corrected region"
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$p_i$")

ax2.grid(True)
ax2.legend()

plt.show()

In [ ]:
print(ui_diag.keys())

In [ ]:
# ---------------------------------------------------------
# Correct p_i using the same bad_mask used for u_i
# ---------------------------------------------------------

p_i_raw = p_i.copy()
p_i_corrected = p_i.copy()

# Good points = points that were NOT flagged for correction
good_mask = (~bad_mask) & np.isfinite(p_i_raw)

# Bad points to replace
replace_mask = bad_mask & np.isfinite(xc)

# Linear interpolation across the bad regions
p_i_corrected[replace_mask] = np.interp(
    xc[replace_mask],
    xc[good_mask],
    p_i_raw[good_mask]
)

# Replace p_i with corrected version
p_i = p_i_corrected

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

# ---------------------------------------------------------
# Interface
# ---------------------------------------------------------

ax1.plot(
    xc,
    h,
    linewidth=1.2,
    label=r"$h$"
)

ax1.scatter(
    xc[bad_mask],
    h[bad_mask],
    s=25,
    color="red",
    zorder=5,
    label=r"$u_i$ corrected region"
)

ax1.set_ylabel(r"$h/h_0$")
ax1.set_title(
    r"Correction of $p_i$ using the $u_i$ anomaly mask"
)
ax1.grid(True)
ax1.legend()


# ---------------------------------------------------------
# Pressure
# ---------------------------------------------------------

# Original pressure
ax2.plot(
    xc,
    p_i_raw,
    linewidth=1.0,
    alpha=0.6,
    label=r"Raw $p_i$"
)

# Corrected pressure
ax2.plot(
    xc,
    p_i_corrected,
    linewidth=1.3,
    label=r"Corrected $p_i$"
)

# Points that were replaced
ax2.scatter(
    xc[bad_mask],
    p_i_corrected[bad_mask],
    s=25,
    color="red",
    zorder=5,
    label="Corrected points"
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$p_i$")

ax2.grid(True)
ax2.legend()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Optimize Savitzky-Golay filter for interfacial pressure
# ---------------------------------------------------------

m_values = range(11, 102, 2)   # odd window lengths
p_values = range(2, 3)        # polynomial orders

best_m_p, best_p_p, best_error_p, results_p = best_savgol_filter(
    xc,
    p_i,
    m_values=m_values,
    p_values=p_values,
    metric="rmse"
)

print("Best Savitzky-Golay parameters for p_i:")
print(f"  m     = {best_m_p}")
print(f"  p     = {best_p_p}")
print(f"  RMSE  = {best_error_p:.6e}")

In [ ]:
# ---------------------------------------------------------
# Apply optimized Savitzky-Golay filter
# ---------------------------------------------------------

p_i_unfiltered = p_i.copy()

p_i_filtered = savgol_filter(
    p_i_unfiltered,
    window_length=best_m_p,
    polyorder=best_p_p
)

# Use filtered pressure from this point onward
p_i = p_i_filtered

Revamping the process for the final term. Let's forget what I have done for pressure so far.

Let's rewrite the interesting pressure term as
$\int_0^h \partial_x p dz = \partial_x \int_0^h p dy - p_i h_x$

Let's obtain the first and second terms in the RHS one by one.

In [ ]:
_, P = depth_integral(x,y,p*f)

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 7),
    sharex=True,
    constrained_layout=True
)

# Interface
ax1.plot(
    xc,
    h,
    linewidth=1.2
)

ax1.set_ylabel(r"$h/h_0$")
ax1.set_title("Interface and depth-integrated pressure")
ax1.grid(True)


# Depth-integrated pressure
ax2.plot(
    xc,
    P,
    linewidth=1.2
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$\int f p\,dy$")
ax2.grid(True)

plt.show()

In [ ]:
p_i, pi_diag = compute_ui(
    p,
    y,
    h,
    xc,
    k=5,
    prominence=0.05,
    distance=10,
    Z_h_threshold=3.0,
    Z_u_region_threshold=2.0,
    return_diagnostics=True
)

In [ ]:
# ---------------------------------------------------------
# Extract pressure diagnostics
# ---------------------------------------------------------

p_i_corrected = p_i
p_i_interpolated = pi_diag["u_i_raw"]

anomaly_indices = pi_diag["anomaly_indices"]
bad_mask = pi_diag["bad_mask"]


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

# ---------------------------------------------------------
# 1. Pressure returned by compute_ui()
# ---------------------------------------------------------

ax1.plot(
    xc,
    p_i_corrected,
    linewidth=1.2,
    label=r"Corrected $p_i$"
)

ax1.set_ylabel(r"$p_i$")
ax1.set_title(
    r"Interfacial pressure from anomaly-correction function"
)

ax1.grid(True)
ax1.legend()


# ---------------------------------------------------------
# 2. Straight interpolated pressure + detected anomalies
# ---------------------------------------------------------

ax2.plot(
    xc,
    p_i_interpolated,
    linewidth=1.2,
    label=r"Interpolated $p_i$"
)

# Actual points classified as anomalies
ax2.scatter(
    xc[anomaly_indices],
    p_i_interpolated[anomaly_indices],
    s=40,
    color="red",
    marker="o",
    zorder=5,
    label="Detected anomalies"
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$p_i$")
ax2.set_title(
    r"Straight interpolated pressure with detected anomalies"
)
# ax2.set_xlim(450,500)
ax2.grid(True)
ax2.legend()

plt.show()

In [ ]:
p_i_unfiltered = p_i.copy()

p_i_filtered = savgol_filter(
    p_i_unfiltered,
    window_length=11,
    polyorder=3
)

# Use filtered pressure from this point onward
# p_i = p_i_unfiltered

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

# ---------------------------------------------------------
# Unfiltered interfacial pressure
# ---------------------------------------------------------

ax1.plot(
    xc,
    p_i_unfiltered,
    linewidth=1.2,
    label=r"Unfiltered $p_i$"
)

ax1.set_ylabel(r"$p_i$")
ax1.set_title("Unfiltered interfacial pressure")
ax1.grid(True)
ax1.legend()


# ---------------------------------------------------------
# Filtered interfacial pressure
# ---------------------------------------------------------

ax2.plot(
    xc,
    p_i_filtered,
    linewidth=1.2,
    label=r"Filtered $p_i$ ($m=11,\ p=2$)"
)

ax2.set_ylabel(r"$p_i$")
ax2.set_title("Savitzky-Golay filtered interfacial pressure")
ax2.grid(True)
ax2.legend()


# ---------------------------------------------------------
# Interface height
# ---------------------------------------------------------

ax3.plot(
    xc,
    h,
    linewidth=1.2,
    label=r"$h$"
)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel(r"$h/h_0$")
ax3.set_title("Interface height")
ax3.set_xlim(350,400)
ax3.grid(True)
ax3.legend()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Pressure-gradient integral using Leibniz rule
#
# ∫_0^h p_x dy = d/dx(∫_0^h p dy) - p_i h_x
# ---------------------------------------------------------

# First term: d/dx ∫_0^h p dy
P_x = first_derivative_x(xc, P)

# Second term: p_i * h_x
pi_hx = p_i_filtered * h_x

# Combined pressure-gradient integral
pressure_gradient_integral = P_x - pi_hx


# ---------------------------------------------------------
# Plot individual terms and their combination
# ---------------------------------------------------------

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

# Term 1
ax1.plot(
    xc,
    P_x,
    linewidth=1.2,
    label=r"$\partial_x \left(\int_0^h p\,dy\right)$"
)

ax1.set_ylabel("Pressure term")
ax1.set_title(r"$\partial_x \left(\int_0^h p\,dy\right)$")
ax1.grid(True)
ax1.legend()


# Term 2
ax2.plot(
    xc,
    -pi_hx,
    linewidth=1.2,
    label=r"$-p_i h_x$"
)

ax2.set_ylabel("Pressure term")
ax2.set_title(r"$-p_i h_x$")
ax2.grid(True)
ax2.legend()


# Combined result
ax3.plot(
    xc,
    pressure_gradient_integral,
    linewidth=1.2,
    label=r"$\int_0^h \partial_x p\,dy$"
)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel("Pressure term")
ax3.set_title(
    r"$\int_0^h \partial_x p\,dy"
)
ax3.set_ylim(-1,1)
ax3.set_xlim(350,400)
ax3.grid(True)
ax3.legend()

plt.show()

In [ ]:
p_i_interpolated = pi_diag["u_i_raw"]

In [ ]:
# P_x should be calculated from the same P = ∫ f p dy
P_x = first_derivative_x(xc, P)

# ---------------------------------------------------------
# Leibniz using raw interfacial pressure
# ---------------------------------------------------------

I_p_raw_pi = (
    P_x
    - p_i_interpolated * h_x
)

# ---------------------------------------------------------
# Leibniz using processed interfacial pressure
# ---------------------------------------------------------

I_p_filtered_pi = (
    P_x
    - p_i_filtered * h_x
)


# ---------------------------------------------------------
# Compare
# ---------------------------------------------------------

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

ax1.plot(
    xc,
    P_x,
    linewidth=1.2,
    label=r"$P_x$"
)

ax1.plot(
    xc,
    -p_i_interpolated * h_x,
    linewidth=1.2,
    label=r"$-p_{i,\mathrm{raw}}h_x$"
)

ax1.set_ylabel("Pressure term")
ax1.set_title("Consistent raw quantities")
ax1.grid(True)
ax1.legend()


ax2.plot(
    xc,
    I_p_raw_pi,
    linewidth=1.0,
    label=r"$P_x-p_{i,\mathrm{raw}}h_x$"
)

ax2.set_ylabel(r"$\int_0^h p_x\,dy$")
ax2.set_title("Leibniz result using raw interfacial pressure")
ax2.grid(True)
ax2.legend()


ax3.plot(
    xc,
    I_p_filtered_pi,
    linewidth=1.0,
    label=r"$P_x-p_{i,\mathrm{filtered}}h_x$"
)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel(r"$\int_0^h p_x\,dy$")
ax3.set_title("Leibniz result using filtered interfacial pressure")
ax3.grid(True)
ax3.legend()
ax3.set_xlim(350,400)


ax2.set_ylim(-1, 1)
ax3.set_ylim(-1,1)
plt.show()

In [ ]:
# ---------------------------------------------------------
# Compare finite-difference and SG derivatives of
# P = ∫_0^h p dy
# ---------------------------------------------------------

dx = np.mean(np.diff(xc))

# Existing finite-difference derivative
P_x_fd = first_derivative_x(xc, P)

# SG derivative directly from P
P_x_sg = savgol_filter(
    P,
    window_length=11,
    polyorder=2,
    deriv=1,
    delta=dx
)

# Leibniz residuals
I_p_fd = P_x_fd - p_i_filtered * h_x
I_p_sg = P_x_sg - p_i_filtered * h_x


fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

# Derivatives of integrated pressure
ax1.plot(
    xc, P_x_fd,
    linewidth=1.0,
    label=r"$P_x$ -- finite difference"
)

ax1.plot(
    xc, P_x_sg,
    linewidth=1.2,
    label=r"$P_x$ -- SG derivative"
)

ax1.set_ylabel(r"$P_x$")
ax1.set_title(
    r"Derivative of $P=\int_0^h p\,dy$"
)
ax1.grid(True)
ax1.legend()


# Difference between derivative methods
ax2.plot(
    xc,
    P_x_fd - P_x_sg,
    linewidth=1.0
)

ax2.set_ylabel(r"$P_{x,FD}-P_{x,SG}$")
ax2.set_title("Difference between derivative estimates")
ax2.grid(True)


# Resulting pressure-gradient integral
ax3.plot(
    xc, I_p_fd,
    linewidth=0.9,
    alpha=0.7,
    label="Finite difference"
)

ax3.plot(
    xc, I_p_sg,
    linewidth=1.2,
    label="SG derivative"
)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel(r"$\int_0^h p_x\,dy$")
ax3.set_title("Leibniz pressure-gradient integral")
ax3.grid(True)
ax3.legend()

ax3.set_xlim(350,400)


ax2.set_ylim(-1, 1)
ax3.set_ylim(-2,2)

plt.show()

In [ ]:
# Leibniz estimate
I_leibniz = P_x_sg - p_i_filtered * h_x

# Previously calculated direct integrals
I_090 = Ip_fraction[0.90]
I_095 = Ip_fraction[0.95]
I_098 = Ip_fraction[0.98]
I_100 = Ip_fraction[1.00]

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

# Direct integration sensitivity
ax1.plot(xc, I_090, label=r"$\int_0^{0.90h}p_x\,dy$")
ax1.plot(xc, I_095, label=r"$\int_0^{0.95h}p_x\,dy$")
ax1.plot(xc, I_098, label=r"$\int_0^{0.98h}p_x\,dy$")
ax1.plot(xc, I_100, label=r"$\int_0^{h}p_x\,dy$", alpha=0.6)

ax1.set_ylabel("Pressure-gradient integral")
ax1.set_title("Direct integration sensitivity")
ax1.grid(True)
ax1.legend()


# Compare full-depth approaches
ax2.plot(
    xc,
    I_leibniz,
    label="Leibniz (SG derivative)",
    linewidth=1.2
)

ax2.plot(
    xc,
    I_100,
    label=r"Direct $\int_0^h p_x\,dy$",
    linewidth=1.0,
    alpha=0.7
)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel(r"$\int p_x\,dy$")
ax2.set_title("Independent full-depth estimates")
ax2.grid(True)
ax2.legend()
ax2.set_xlim(350,400)
plt.show()

In [ ]:
I_ref = Ip_fraction[1.00]

for alpha in [0.90, 0.95, 0.98]:

    I_alpha = Ip_fraction[alpha]

    valid = np.isfinite(I_alpha) & np.isfinite(I_ref)

    rel_L2 = (
        np.linalg.norm(I_alpha[valid] - I_ref[valid])
        / np.linalg.norm(I_ref[valid])
    )

    max_diff = np.max(
        np.abs(I_alpha[valid] - I_ref[valid])
    )

    print(
        f"alpha = {alpha:.2f}: "
        f"relative L2 = {rel_L2:.4e}, "
        f"max |difference| = {max_diff:.4e}"
    )

In [ ]:
I_p_direct = Ip_fraction[1.00].copy()

I_p_filtered = savgol_filter(
    I_p_direct,
    window_length=11,
    polyorder=2
)

In [ ]:
filter_residual = I_p_direct - I_p_filtered

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(14, 8),
    sharex=True,
    constrained_layout=True
)

ax1.plot(
    xc, I_p_direct,
    linewidth=0.8,
    alpha=0.6,
    label=r"Raw $\int_0^h p_x\,dy$"
)

ax1.plot(
    xc, I_p_filtered,
    linewidth=1.3,
    label=r"SG filtered ($m=11,\ p=2$)"
)

ax1.set_ylabel(r"$\int_0^h p_x\,dy$")
ax1.set_title("Depth-integrated pressure gradient")
ax1.grid(True)
ax1.legend()


ax2.plot(
    xc,
    filter_residual,
    linewidth=0.8
)

ax2.axhline(0, color="black", linewidth=0.7)

ax2.set_xlabel(r"$x/h_0$")
ax2.set_ylabel("Raw - filtered")
ax2.set_title("Pressure-gradient content removed by filtering")
ax2.grid(True)
ax2.set_xlim(350,400)
plt.show()

In [ ]:
windows = [5, 7, 9, 11, 15, 21]

plt.figure(figsize=(14, 5))

plt.plot(
    xc,
    I_p_direct,
    linewidth=0.7,
    alpha=0.4,
    label="Raw"
)

for m in windows:

    I_temp = savgol_filter(
        I_p_direct,
        window_length=m,
        polyorder=2
    )

    plt.plot(
        xc,
        I_temp,
        linewidth=1.0,
        label=rf"$m={m}$"
    )

plt.xlabel(r"$x/h_0$")
plt.ylabel(r"$\int_0^h p_x\,dy$")
plt.xlim(350,400)
plt.title("Savitzky-Golay window sensitivity")
plt.grid(True)
plt.legend()
plt.show()

Before making a final decision on the filtering, let's first see what the pressure correction looks like without filtering.

In [ ]:

# ---------------------------------------------------------
# Raw DNS pressure contribution
#
# P_x,DNS = - ∫_0^h dp/dx dz
# ---------------------------------------------------------

P_x_DNS_raw = -Ip_fraction[1.00].copy()
h_xxx = first_derivative_x(xc, h_xx)
We = Re * Ca
Fr2 = Re * np.sin(beta) / 3
# ---------------------------------------------------------
# Lavalle representation
#
# P_x,Lav =
#       h h_xxx / We
#     - (cos(beta) / Fr^2) h h_x
#     + q_xx / Re
# ---------------------------------------------------------

P_capillary_Lav = (
    h * h_xxx / We
)

P_hydrostatic_Lav = (
    -np.cos(beta) / Fr2 * h * h_x
)

P_viscous_Lav = (
    q_xx / Re
)

P_x_Lav = (
    P_capillary_Lav
    + P_hydrostatic_Lav
    + P_viscous_Lav
)


# ---------------------------------------------------------
# Raw DNS - Lavalle correction
# ---------------------------------------------------------

delta_P_x_raw = (
    P_x_DNS_raw
    - P_x_Lav
)

In [ ]:
beta*180/3.14

In [ ]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(
    4, 1,
    figsize=(14, 12),
    sharex=True,
    constrained_layout=True
)

# DNS pressure contribution
ax1.plot(
    xc,
    P_x_DNS_raw,
    linewidth=1.0,
    label=r"$\mathcal{P}_{x,\mathrm{DNS}}$"
)

ax1.set_ylabel("Pressure term")
ax1.set_title(
    r"DNS: $-\int_0^h \partial_x p\,dz$"
)
ax1.grid(True)
ax1.legend()


# Individual Lavalle contributions
ax2.plot(
    xc,
    P_capillary_Lav,
    linewidth=1.0,
    label=r"$h h_{xxx}/We$"
)

ax2.plot(
    xc,
    P_hydrostatic_Lav,
    linewidth=1.0,
    label=r"$-(\cos\beta/Fr^2)hh_x$"
)

ax2.plot(
    xc,
    P_viscous_Lav,
    linewidth=1.0,
    label=r"$q_{xx}/Re$"
)

ax2.set_ylabel("Pressure term")
ax2.set_title("Lavalle contributions")
ax2.grid(True)
ax2.legend()


# Total Lavalle representation
ax3.plot(
    xc,
    P_x_Lav,
    linewidth=1.2,
    label=r"$\mathcal{P}_{x,\mathrm{Lav}}$"
)

ax3.set_ylabel("Pressure term")
ax3.set_title("Total Lavalle representation")
ax3.grid(True)
ax3.legend()


# Raw correction
ax4.plot(
    xc,
    delta_P_x_raw,
    linewidth=1.0,
    label=r"$\Delta\mathcal{P}_x^{\mathrm{raw}}$"
)

ax4.axhline(
    0.0,
    linewidth=0.7,
    color="black"
)

ax4.set_xlabel(r"$x/h_0$")
ax4.set_ylabel("Correction")
ax4.set_title(
    r"$\mathcal{P}_{x,\mathrm{DNS}}"
    r"-\mathcal{P}_{x,\mathrm{Lav}}$"
)

ax4.grid(True)
ax4.legend()
ax4.set_xlim(350,400)
plt.show()

In [ ]:
# ---------------------------------------------------------
# Minimal filtering of raw pressure correction
# ---------------------------------------------------------

delta_P_x_filtered = savgol_filter(
    delta_P_x_raw,
    window_length=9,
    polyorder=2
)

# Amount removed by filtering
delta_P_x_removed = (
    delta_P_x_raw - delta_P_x_filtered
)


# ---------------------------------------------------------
# Compare raw and filtered quantities
# ---------------------------------------------------------

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

# Raw correction
ax1.plot(
    xc,
    delta_P_x_raw,
    linewidth=1.0,
    label=r"Raw $\Delta\mathcal{P}_x$"
)

ax1.axhline(0.0, color="black", linewidth=0.7)

ax1.set_ylabel(r"$\Delta\mathcal{P}_x$")
ax1.set_title("Raw pressure correction")
ax1.grid(True)
ax1.legend()


# Raw vs filtered
ax2.plot(
    xc,
    delta_P_x_raw,
    linewidth=0.8,
    alpha=0.5,
    label=r"Raw $\Delta\mathcal{P}_x$"
)

ax2.plot(
    xc,
    delta_P_x_filtered,
    linewidth=1.3,
    label=r"Filtered $\Delta\mathcal{P}_x$ ($m=5,\ p=2$)"
)

ax2.axhline(0.0, color="black", linewidth=0.7)

ax2.set_ylabel(r"$\Delta\mathcal{P}_x$")
ax2.set_title("Raw and minimally filtered pressure correction")
ax2.grid(True)
ax2.legend()


# Content removed by filter
ax3.plot(
    xc,
    delta_P_x_removed,
    linewidth=1.0,
    label=r"Raw $-$ filtered"
)

ax3.axhline(0.0, color="black", linewidth=0.7)

ax3.set_xlabel(r"$x/h_0$")
ax3.set_ylabel("Difference")
ax3.set_title("Content removed by filtering")
ax3.grid(True)
ax3.legend()
ax3.set_xlim(350,400)
plt.show()